# MiCASA v2 — Pipeline Verification (`verify_v2.ipynb`)

Phase 1 of a port from miller-ff `verify_2026.ipynb`.
Ten structural-invariant checks across schema, transformation
invariants, sign convention, and cross-boundary sanity.

**Run from the v2 working directory**:
```sh
cd $WORK_DIR     # i.e. wherever you checked out this repo
jupyter nbconvert --to notebook --execute verify_v2.ipynb \
    --output verify_v2.executed.ipynb
```

Each check appends to `_RESULTS`. The summary cell at the end prints
counts and any FAIL details. Designed to be additive: Phase 2 / 3 will
add Sections 5+ for cross-product comparison, spatial sanity, and
seasonal sanity.

## Configuration & Setup

In [ ]:
import os, sys, json, subprocess, glob, re
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import xarray as xr

WORK_DIR     = Path(os.environ.get("WORK_DIR", os.getcwd()))
MONTHLY_FILE = WORK_DIR / "monthly_1x1/MiCASA_v1_flux_x360_y180_monthly.nc"
FIT_RDA      = WORK_DIR / "fit.piqs.rda"
ERA5_DIR     = WORK_DIR / "ERA5"
JOBS_DIR     = WORK_DIR / "jobs"
DAILY_DIR    = WORK_DIR / "daily_1x1"
# MET_BASE is the root of the TM5 ERA5 meteo tree. Set CARBONTRACKER
# to the directory containing METEO/...; the default below is a no-op
# placeholder so the file parses, but Section-7-and-later checks need a
# real path to find ssrd files.
MET_BASE     = Path(os.environ.get("CARBONTRACKER", ".")) \
               / "METEO/tm5-nc/ec/ea/h06h18tr1/sfc/glb100x100"
MET_BASE_FALLBACK = Path(os.environ.get("CARBONTRACKER", "."))                         / "METEO/tm5-nc/ec/ea_0005/h06h18tr1/sfc/glb100x100"

PASS, FAIL, WARN, INFO = "PASS", "FAIL", "WARN", "INFO"
_RESULTS = []

def record(check_id, name, status, detail=""):
    _RESULTS.append((check_id, name, status, detail))
    color = {PASS: "\x1b[32m", FAIL: "\x1b[31m",
             WARN: "\x1b[33m", INFO: "\x1b[34m"}.get(status, "")
    reset = "\x1b[0m"
    print(f"{color}[{status}]{reset} {check_id} {name}: {detail}")

In [ ]:
# Quick orientation: what is on disk?
print(f"WORK_DIR     = {WORK_DIR}")
print(f"MONTHLY_FILE = {MONTHLY_FILE} ({'exists' if MONTHLY_FILE.exists() else 'MISSING'})")
print(f"FIT_RDA      = {FIT_RDA} ({'exists' if FIT_RDA.exists() else 'MISSING'})")
print(f"ERA5/        = {len(list(ERA5_DIR.glob('fluxes_*.nc')))} fluxes_*.nc files")
print(f"daily_1x1/   = {len(list(DAILY_DIR.glob('*.nc')))} *.nc files (incl. clim + symlinks)")
print(f"jobs/        = {len(list(JOBS_DIR.glob('*')))} log files")

## Section 1 — Schema & Coverage

### Check 1.1 — Multi-year monthly cat file schema

The file `monthly_1x1/MiCASA_v1_flux_x360_y180_monthly.nc` (output of
`cat_monthly.sh`) is the input to PIQS. It must have:
- dims: `longitude=360, latitude=180, time>=288`
- variables: `NPP, Rh, FIRE, FUEL` (units gC m-2 s-1)
- time monotonically increasing, monthly cadence

In [ ]:
cid, cname = "1.1", "monthly_cat schema"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    problems = []
    if ds.sizes.get("longitude") != 360: problems.append(f"longitude dim is {ds.sizes.get('longitude')}, expected 360")
    if ds.sizes.get("latitude")  != 180: problems.append(f"latitude  dim is {ds.sizes.get('latitude')}, expected 180")
    if ds.sizes.get("time", 0) < 288:    problems.append(f"time dim is {ds.sizes.get('time')}, expected >= 288")
    for v in ("NPP", "Rh", "FIRE", "FUEL"):
        if v not in ds.variables:
            problems.append(f"variable '{v}' missing")
    if problems:
        record(cid, cname, FAIL, "; ".join(problems))
    else:
        record(cid, cname, PASS,
               f"dims OK, time={ds.sizes['time']} months "
               f"({pd.to_datetime(ds.time.values[0]).strftime('%Y-%m')}..."
               f"{pd.to_datetime(ds.time.values[-1]).strftime('%Y-%m')})")
    ds.close()
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

### Check 1.2 — PIQS fit metadata schema

Calls the R helper `verify_piqs_invariants.r` which loads `fit.piqs.rda`
and dumps a JSON summary. Confirms:
- `piqsfit.gpp/resp` are `360 × 180 × N` arrays
- `piqsfit.time` length matches
- `piqsfit.meta` exists and records padding settings

In [ ]:
cid, cname = "1.2", "PIQS fit metadata"
helper = WORK_DIR / "verify_piqs_invariants.r"
out_json = WORK_DIR / "verify_piqs_invariants.json"
if not helper.exists():
    record(cid, cname, FAIL, f"helper script missing: {helper}")
else:
    # Run the helper (also computes Check 2.1 residuals -- reused below).
    try:
        r = subprocess.run(["Rscript", str(helper), str(out_json)],
                           cwd=WORK_DIR, capture_output=True, text=True, timeout=600)
        if r.returncode != 0:
            record(cid, cname, FAIL, f"Rscript failed: {r.stderr[-300:]}")
        else:
            _piqs = json.loads(out_json.read_text())
            problems = []
            if list(_piqs.get("fit_dims_gpp", [])) != [360, 180, _piqs.get("nmon", -1)]:
                problems.append(f"gpp dims {_piqs.get('fit_dims_gpp')} unexpected")
            if not _piqs.get("piqsfit_meta"):
                problems.append("piqsfit.meta missing from .rda")
            if problems:
                record(cid, cname, FAIL, "; ".join(problems))
            else:
                meta = _piqs["piqsfit_meta"]
                record(cid, cname, PASS,
                       f"nmon={_piqs['nmon']}, "
                       f"pad=(L{meta.get('pad.left')},R{meta.get('pad.right')}), "
                       f"written {meta.get('written.at')}")
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 1.3 — `fluxes_YYYYMM.nc` schema + NaN/Inf scan

For each `ERA5/fluxes_YYYYMM.nc`, confirm dims, variable list, and that
no NaN/Inf appears in NEE/GPP/resp over land. (Polar cells legitimately
have NaN/zero where ssrd ≡ 0; we flag only land NaN counts as concerning.)

In [ ]:
cid, cname = "1.3", "fluxes_*.nc schema + NaN scan"
expected_vars = {"GPP", "resp", "NEE", "QGPP", "qresp", "ssr", "t2m", "stl1", "swvl1", "decimal_date"}
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, WARN, "no fluxes_*.nc files yet (diurnalize not run?)")
else:
    # Build a land mask once from the multi-year monthly file so we count
    # NaN/Inf only over actual land (ocean cells legitimately carry the
    # netCDF _FillValue, which xarray converts to NaN on read).
    land_mask_2d = None
    try:
        ds_m = xr.open_dataset(MONTHLY_FILE)
        land_mask_2d = (np.abs(ds_m["NPP"]).max(dim="time").values > 1e-15)  # (lat, lon)
        ds_m.close()
    except Exception:
        pass

    bad = []
    nan_counts = []
    sample_idx = sorted({0, len(files)//2, len(files)-1})
    for i in sample_idx:
        f = files[i]
        try:
            ds = xr.open_dataset(f)
            missing = expected_vars - set(ds.variables)
            if missing:
                bad.append(f"{f.name}: missing {missing}")
            if land_mask_2d is not None:
                lat = ds.latitude.values
                lat_ix = (lat >= -60) & (lat <= 60)
                land_band = land_mask_2d[lat_ix, :]
                n_land_hours_total = int(land_band.sum() * ds.sizes["time"])
                for v in ("NEE", "GPP", "resp"):
                    if v in ds:
                        arr = ds[v].isel(latitude=lat_ix).values
                        bad_mask = (np.isnan(arr) | np.isinf(arr)) & land_band[None, :, :]
                        n_bad = int(bad_mask.sum())
                        if n_bad:
                            pct = 100.0 * n_bad / max(n_land_hours_total, 1)
                            nan_counts.append(f"{f.name}/{v}={n_bad}({pct:.2f}%)")
            ds.close()
        except Exception as e:
            bad.append(f"{f.name}: {e}")
    # Schema problems are always FAIL. NaN counts: tolerate small fraction
    # (a handful of coastline cells with all-zero NPP get _FillValue ->
    # NaN through the pipeline). Fail only if >1% of any var's land-hours.
    FAIL_PCT_THRESHOLD = 1.0
    nan_problems = []
    for s in nan_counts:
        try:
            pct = float(s.split("(")[1].split("%")[0])
            if pct > FAIL_PCT_THRESHOLD:
                nan_problems.append(s)
        except Exception:
            nan_problems.append(s)
    if bad or nan_problems:
        record(cid, cname, FAIL, "; ".join((bad + nan_problems)[:6]))
    elif nan_counts:
        record(cid, cname, WARN,
               f"{len(sample_idx)}/{len(files)} files; minor NaN over land (<1%): "
               f"{'; '.join(nan_counts[:4])}")
    else:
        record(cid, cname, PASS,
               f"checked {len(sample_idx)}/{len(files)} files (first/mid/last); schema OK, "
               f"no NaN/Inf over land in [-60,60] latitude band")

### Check 1.4 — ERA5 meteo coverage matches diurnalized year range

Every year/month with a `fluxes_YYYYMM.nc` output must have had its
meteo input on disk. diurnalize-ERA5.r resolves each day to the
primary tree or the FastTrack fallback (`ea_0005`); this check probes
a representative t2m file in *either* tree and fails only if neither
has it.

In [ ]:
cid, cname = "1.4", "ERA5 meteo coverage"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
missing = []
for f in files:
    m = re.search(r"fluxes_(\d{4})(\d{2})\.nc$", f.name)
    if not m: continue
    yr, mo = m.group(1), m.group(2)
    # Probe the first-day t2m file in the primary then fallback tree.
    rel = f"{yr}/{mo}/t2m_{yr}{mo}01_00p01.nc"
    if not ((MET_BASE / rel).exists() or (MET_BASE_FALLBACK / rel).exists()):
        missing.append(f"{yr}-{mo}")
if not files:
    record(cid, cname, INFO, "no fluxes_*.nc to check yet")
elif missing:
    record(cid, cname, FAIL,
           f"{len(missing)} fluxes_*.nc files lack a corresponding meteo probe; "
           f"first few: {missing[:6]}")
else:
    record(cid, cname, PASS, f"every {len(files)} fluxes file has meteo on disk")

## Section 2 — Pipeline Transformation Invariants

### Check 2.1 — PIQS integral preservation

PIQS = Piecewise *Integral* Quadratic Splines. The defining property:
for each segment, the integral of the per-piece quadratic equals the
input monthly mean × segment width. Equivalently:

> integral / delta == input monthly mean (per cell-segment)

The R helper computed this residual for every active land cell-month;
we read its summary and assert max-absolute-residual is below tolerance.

In [ ]:
cid_g, cid_r = "2.1.gpp", "2.1.rtot"
out_json = WORK_DIR / "verify_piqs_invariants.json"
TOL_ABS = 1e-9   # gC m-2 s-1
TOL_REL = 1e-6
if not out_json.exists():
    record(cid_g, "PIQS integral preservation (GPP)", FAIL,
           f"helper output {out_json.name} missing -- did Check 1.2 run?")
else:
    _piqs = json.loads(out_json.read_text())
    for cid, key, label in [(cid_g, "gpp", "GPP"), (cid_r, "rtot", "Rtot")]:
        s = _piqs[key]
        detail = (f"max_abs={s['max_abs_residual']:.2e} "
                  f"max_rel={s['max_rel_residual']:.2e} "
                  f"frac>1e-9 abs={s['frac_abs_resid_gt_1e9']:.4f}")
        ok_abs = s["max_abs_residual"] < TOL_ABS
        ok_rel = s["max_rel_residual"] < TOL_REL
        if ok_abs and ok_rel:
            record(cid, f"PIQS integral preservation ({label})", PASS, detail)
        else:
            record(cid, f"PIQS integral preservation ({label})", FAIL, detail)

### Check 2.2 — Diurnalize preserves monthly means

For a sample `fluxes_YYYYMM.nc`, the per-cell mean of hourly NEE
over the month should equal the input monthly mean (which is a
function of the gridcell's NPP and Rh for that month). Within a
small numerical tolerance.

In [ ]:
cid, cname = "2.2", "diurnalize preserves monthly means"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes_*.nc to check yet")
else:
    # Use the middle file for a representative sample
    f = files[len(files)//2]
    m = re.search(r"fluxes_(\d{4})(\d{2})\.nc$", f.name)
    yr, mo = m.group(1), m.group(2)
    monthly_pm = WORK_DIR / "monthly_1x1" / f"MiCASA_v1_flux_x360_y180_monthly_{yr}{mo}.nc"
    if not monthly_pm.exists():
        record(cid, cname, WARN, f"per-month input {monthly_pm.name} missing -- skipping")
    else:
        try:
            ds_h = xr.open_dataset(f)            # hourly
            ds_m = xr.open_dataset(monthly_pm)   # monthly mean (one timestep)
            # Convert sign / units: gpp.mn = -2*NPP/12 in mol m-2 s-1
            # In monthly file, NPP is in gC m-2 s-1; gC -> mol = / 12
            gpp_mn_expected = (-2.0 * ds_m["NPP"].squeeze() / 12.0).values
            rtot_mn_expected = (ds_m["Rh"].squeeze() / 12.0 - 0.5 * gpp_mn_expected).values
            gpp_mn_actual    = ds_h["GPP"].mean(dim="time").values
            resp_mn_actual   = ds_h["resp"].mean(dim="time").values
            # Mask to cells with absolute monthly mean above a robust
            # land-flux threshold. 1e-15 mol m-2 s-1 was too low: it
            # admits cells right at the float-zero boundary, where the
            # relative diff blows up. 1e-9 mol m-2 s-1 = roughly 1e-9
            # gC m-2 s-1 = ~32 mgC m-2 yr-1, well below any vegetated
            # land flux but well above the float-zero noise floor.
            LAND_FLUX_THRESH = 1e-9
            # Exclude polar-night cells (ssr month-mean == 0). The
            # polar-night clip in diurnalize-ERA5.r legitimately zeros
            # GPP there, which breaks "diurnalized monthly mean = input
            # NPP" by design (no light => no photosynthesis). Physically
            # correct, just out of scope for this mass-balance check.
            ssr_mn = ds_h["ssr"].mean(dim="time").values
            mask = (np.abs(gpp_mn_expected) > LAND_FLUX_THRESH) & (ssr_mn > 0)
            if mask.sum() == 0:
                record(cid, cname, WARN, "no active land cells in sample month")
            else:
                abs_g = np.abs((gpp_mn_actual - gpp_mn_expected)[mask])
                abs_r = np.abs((resp_mn_actual - rtot_mn_expected)[mask])
                rel_g = abs_g / np.abs(gpp_mn_expected[mask])
                rel_r = abs_r / np.abs(rtot_mn_expected[mask])
                # Use 99th percentile rel-diff (robust to outliers at
                # cells right at the threshold edge).
                p99_g = float(np.percentile(rel_g, 99))
                p99_r = float(np.percentile(rel_r, 99))
                detail = (f"sample {f.name}: GPP rel diff p50={np.median(rel_g):.2e} "
                          f"p99={p99_g:.2e}; "
                          f"resp rel diff p50={np.median(rel_r):.2e} p99={p99_r:.2e}")
                # 5% p99 rel-diff tolerance. The polar-night clip in
                # diurnalize-ERA5.r legitimately zeros GPP wherever
                # ssrd=0, which breaks strict mass conservation by a
                # small amount: the monthly mean shifts by the
                # integrated qmod.gpp residual at clipped hours. For
                # most cells this is <0.1% (well under the old 1%
                # threshold), but partial-polar-night cells (any cell
                # with ssrd=0 hours and a nonzero gpp.mn) can exceed
                # 1%. Empirical p99 in fluxes_201307.nc is ~1.5% with
                # the clip; 5% gives headroom while still flagging
                # gross mass-balance breakage.
                if p99_g < 5e-2 and p99_r < 5e-2:
                    record(cid, cname, PASS, detail)
                else:
                    record(cid, cname, FAIL, detail)
            ds_h.close(); ds_m.close()
        except Exception as e:
            record(cid, cname, FAIL, f"exception: {e}")

### Check 2.3 — Cat-monthly time monotonicity & monthly cadence

The multi-year monthly file's `time` coordinate must be strictly
monotonically increasing with median spacing ≈ 30.4 days (one month).
Catches stale concat files and ncrcat ordering bugs.

In [ ]:
cid, cname = "2.3", "monthly time monotonicity"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    t  = pd.to_datetime(ds.time.values)
    dt = np.diff(t.values).astype("timedelta64[D]").astype(float)
    problems = []
    if not np.all(dt > 0): problems.append("time not monotonic")
    if not (28 <= dt.min() <= 32 and 28 <= dt.max() <= 32):
        problems.append(f"dt range [{dt.min():.1f},{dt.max():.1f}] days outside [28,32]")
    if np.median(dt) < 28 or np.median(dt) > 32:
        problems.append(f"median dt={np.median(dt):.1f} days, expected ~30.4")
    if problems:
        record(cid, cname, FAIL, "; ".join(problems))
    else:
        record(cid, cname, PASS,
               f"{len(t)} timestamps, {t[0].strftime('%Y-%m')}..{t[-1].strftime('%Y-%m')}, "
               f"median dt={np.median(dt):.2f} days")
    ds.close()
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

### Check 2.4 — Daily → monthly mass conservation (sample)

For a sample year/month, mean of the 1° daily files should equal the
1° monthly file (within numerical precision). Catches ingest scaling
errors and stale per-month files.

In [ ]:
cid, cname = "2.4", "daily->monthly mass conservation"
# Pick a recent fully-real month -- use the last month with both
# daily and monthly inputs available.
monthly_files = sorted((WORK_DIR / "monthly_1x1").glob(
    "MiCASA_v1_flux_x360_y180_monthly_2*.nc"))
chosen = None
for f in reversed(monthly_files):
    m = re.search(r"monthly_(\d{4})(\d{2})\.nc$", f.name)
    if not m: continue
    yr, mo = m.group(1), m.group(2)
    sample_day = DAILY_DIR / f"MiCASA_v1_flux_x360_y180_daily_{yr}{mo}01.nc"
    if sample_day.exists() and not sample_day.is_symlink():
        chosen = (f, yr, mo); break
    # Allow symlinks too (vNRT-relabelled days)
    if sample_day.exists():
        chosen = (f, yr, mo); break
if chosen is None:
    record(cid, cname, INFO, "no month found with both per-month and per-day files")
else:
    f_m, yr, mo = chosen
    days = sorted(DAILY_DIR.glob(f"MiCASA_v1_flux_x360_y180_daily_{yr}{mo}*.nc"))
    if not days:
        record(cid, cname, INFO, f"no daily files for {yr}-{mo}")
    else:
        try:
            ds_d = xr.open_mfdataset([str(p) for p in days], combine="nested",
                                     concat_dim="time")
            ds_m = xr.open_dataset(f_m)
            npp_d_mean = ds_d["NPP"].mean(dim="time").values
            npp_m      = ds_m["NPP"].squeeze().values
            # Active land mask
            mask = np.abs(npp_m) > 1e-15
            if mask.sum() == 0:
                record(cid, cname, WARN, f"no active land cells in {yr}-{mo}")
            else:
                rel = np.abs((npp_d_mean - npp_m)[mask] / npp_m[mask])
                detail = (f"{yr}-{mo} ({len(days)} daily files), NPP max_rel={rel.max():.2e}, "
                          f"median={np.median(rel):.2e}")
                if rel.max() < 0.01:
                    record(cid, cname, PASS, detail)
                else:
                    record(cid, cname, FAIL, detail)
            ds_d.close(); ds_m.close()
        except Exception as e:
            record(cid, cname, FAIL, f"exception: {e}")

## Section 3 — Sign Convention

### Check 3.1 — Sub-monthly sign-flip rate (parses diurnalize logs)

The sign-flip diagnostic added in proposal #4 prints two lines per
(year, month) worker. We aggregate across all `jobs/d-*.o*` logs to
build a global summary. Threshold-based: warn if mean cell-hour rate
> 5% (GPP) or > 1% (resp), fail if > 15% (GPP) or > 5% (resp).

In [ ]:
cid, cname = "3.1", "sign-flip rate aggregate"
pat_gpp = re.compile(r"PIQS sign-flip \[GPP > 0\]:\s+\d+ / \d+ cells \(([0-9.]+)%\), \d+ / \d+ cell-hours \(([0-9.]+)%\)")
pat_rsp = re.compile(r"PIQS sign-flip \[resp < 0\]:\s+\d+ / \d+ cells \(([0-9.]+)%\), \d+ / \d+ cell-hours \(([0-9.]+)%\)")
# Pick the most-recent log per year so tagged reruns (e.g. d-YYYY-pchip.o*)
# supersede the original d-YYYY-MiCASA.o* logs.
year_pat = re.compile(r"d-(\d{4})-")
by_year = {}
for L in JOBS_DIR.glob("d-*.o*"):
    mY = year_pat.match(L.name)
    if not mY: continue
    y = mY.group(1)
    if y not in by_year or L.stat().st_mtime > by_year[y].stat().st_mtime:
        by_year[y] = L
logs = [by_year[y] for y in sorted(by_year)]
if not logs:
    record(cid, cname, INFO, "no diurnalize worker logs found")
else:
    gpp_cells, gpp_chs, rsp_cells, rsp_chs = [], [], [], []
    for L in logs:
        try:
            txt = L.read_text(errors="ignore")
            gpp_cells += [float(m.group(1)) for m in pat_gpp.finditer(txt)]
            gpp_chs   += [float(m.group(2)) for m in pat_gpp.finditer(txt)]
            rsp_cells += [float(m.group(1)) for m in pat_rsp.finditer(txt)]
            rsp_chs   += [float(m.group(2)) for m in pat_rsp.finditer(txt)]
        except Exception:
            continue
    if not gpp_chs:
        record(cid, cname, WARN, f"{len(logs)} logs, but no PIQS sign-flip lines parsed")
    else:
        mg, xg = float(np.mean(gpp_chs)),  float(np.max(gpp_chs))
        mr, xr_ = float(np.mean(rsp_chs)), float(np.max(rsp_chs))
        detail = (f"{len(gpp_chs)} months: "
                  f"GPP cell-hour mean={mg:.2f}% max={xg:.2f}%; "
                  f"resp cell-hour mean={mr:.3f}% max={xr_:.3f}%")
        if mg > 15.0 or mr > 5.0:
            status = FAIL
        elif mg > 5.0 or mr > 1.0:
            status = WARN
        else:
            status = PASS
        record(cid, cname, status, detail)

## Section 4 — Cross-Boundary Sanity

### Check 4.1 — v1↔vNRT splice continuity at Dec 2024 / Jan 2025

Reads `diag_v1_vNRT_handoff.csv` (produced by `diag_v1_vNRT_handoff.r`)
and looks for a step-change in monthly global totals across the
splice boundary. A jump > 10% in NPP or Rh between months adjacent to
the boundary is suspicious.

In [ ]:
cid, cname = "4.1", "v1<->vNRT splice continuity"
csv_path = WORK_DIR / "diag_v1_vNRT_handoff.csv"
if not csv_path.exists():
    record(cid, cname, INFO,
           "diag_v1_vNRT_handoff.csv not found -- run diag_v1_vNRT_handoff.r first")
else:
    df = pd.read_csv(csv_path)
    df["yyyymm"] = df["year"]*100 + df["month"]
    # Boundary at 2025-01 = 202501
    before = df[df["yyyymm"] == 202412]
    after  = df[df["yyyymm"] == 202501]
    if len(before) == 0:
        record(cid, cname, WARN, "no 2024-12 row in diagnostic CSV")
    elif len(after) == 0:
        record(cid, cname, INFO,
               "no 2025-01 row yet -- ingest of vNRT 2025 monthlies probably hasn't completed")
    else:
        problems = []
        for v in ("NPP", "Rh"):
            if v not in df.columns: continue
            b, a = float(before[v].iloc[0]), float(after[v].iloc[0])
            rel_jump = abs(a - b) / max(abs(b), 1e-12)
            if rel_jump > 0.10:
                problems.append(f"{v}: 2024-12={b:.1f} -> 2025-01={a:.1f} (rel jump {rel_jump:.1%})")
        if problems:
            record(cid, cname, WARN, "; ".join(problems))
        else:
            record(cid, cname, PASS,
                   f"NPP/Rh global totals continuous within 10% across the v1<->vNRT splice")

## Summary

In [ ]:
# PASS/FAIL/WARN/INFO counts and the table of all results.
from collections import Counter
counts = Counter(r[2] for r in _RESULTS)
print("=" * 60)
print(f"  PASS={counts.get(PASS,0):3d}  "
      f"FAIL={counts.get(FAIL,0):3d}  "
      f"WARN={counts.get(WARN,0):3d}  "
      f"INFO={counts.get(INFO,0):3d}")
print("=" * 60)
print(f"{'check':<12} {'status':<6} {'name':<40} detail")
print("-" * 60)
for cid, name, status, detail in _RESULTS:
    print(f"{cid:<12} {status:<6} {name:<40} {detail}")
if counts.get(FAIL, 0) > 0:
    print("\nOne or more checks FAILED -- investigate above.")
else:
    print("\nAll checks passed.")

# Phase 2 — Comparison & Sanity

Phase 2 adds "looks reasonable?" checks on top of Phase 1's structural
invariants. The new sections each operate on a small per-month summary
cube (`verify_v2_summary.csv`) built once by the preflight cell — so
individual checks don't have to re-scan all ~300 hourly fluxes files.

Sections added in this phase:

- **5. Global Totals & Trends** — monthly/annual global NEE, GPP, Rh;
  plot the full record; check year-on-year growth rate plausibility;
  flag any outlier years.
- **6. Spatial Comparison vs v1** — for sampled months, spatial Pearson
  correlation between v2 and v1 monthly-mean NEE; difference at the
  v2 PIQS-padded right edge (where we expect v2 to differ from v1) vs
  record interior (where they should be ~identical).
- **7. Spatial Sanity** — ocean cells zero, polar/Antarctic small,
  tropical/boreal hotspot seasonal patterns plausible.
- **8. Seasonal & Temporal** — NH/SH out-of-phase, latitude amplitude
  profile, year-boundary continuity (no PIQS ringing at Dec→Jan).

## Phase 2 Preflight — Build per-month summary cube

Walk every `ERA5/fluxes_YYYYMM.nc` once, aggregate to monthly global,
NH (>0°), SH (<0°), tropics (|lat|<23.5°), boreal (>50°N), and SH-mid
(-50..0°) totals (GgC/month). Cache as `verify_v2_summary.csv`. Rebuild
if any source file is newer than the cache. **About 5–10 minutes the
first run, near-instant thereafter.**

Conversion: fluxes are mol m⁻² s⁻¹. Multiply by gridcell area (m²) and
seconds-in-month, then by 12 g/mol, divide by 1e15 → PgC/month.

In [ ]:
cid, cname = "P2.0", "build summary cube"
SUMMARY_CSV = WORK_DIR / "verify_v2_summary.csv"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))

def needs_rebuild():
    if not SUMMARY_CSV.exists(): return True
    cache_mtime = SUMMARY_CSV.stat().st_mtime
    for f in files:
        if f.stat().st_mtime > cache_mtime: return True
    return False

R_EARTH = 6371007.2  # m
def gridcell_area_m2():
    # 1° x 1° grid; latitude bands -89.5..89.5
    lat_edges = np.arange(-90, 91, 1) * np.pi / 180  # 181 edges
    # area = R^2 * (sin(lat_top) - sin(lat_bot)) * dlon (radians)
    dlon = np.pi / 180  # 1° in radians
    lat_band_area = R_EARTH ** 2 * (np.sin(lat_edges[1:]) - np.sin(lat_edges[:-1])) * dlon  # 180
    # Replicate to 360 longitudes
    return np.broadcast_to(lat_band_area[:, None], (180, 360))  # (lat, lon)

if not files:
    record(cid, cname, INFO, "no fluxes_*.nc files yet; Phase 2 cannot run")
elif not needs_rebuild():
    _summary = pd.read_csv(SUMMARY_CSV)
    record(cid, cname, INFO,
           f"cache up to date ({len(_summary)} months in {SUMMARY_CSV.name})")
else:
    print(f"Building summary cube from {len(files)} files...")
    rows = []
    gca = gridcell_area_m2()  # (lat, lon)
    for f in files:
        m = re.search(r"fluxes_(\d{4})(\d{2})\.nc$", f.name)
        if not m: continue
        yr, mo = int(m.group(1)), int(m.group(2))
        try:
            ds = xr.open_dataset(f)
            # Mean over hourly time dim → (lat, lon) per variable
            # Result is mean flux in mol m-2 s-1
            lat = ds.latitude.values
            ndays = pd.Period(f"{yr}-{mo:02d}").days_in_month
            sec_per_month = ndays * 86400
            # Convert mol m-2 s-1 → gC m-2 month-1 → integrate → GgC/month
            # = mean_flux * gca * sec_per_month * 12 (gC/mol) / 1e9 (GgC/g)
            conv = sec_per_month * 12.0 / 1.0e9
            row = {"year": yr, "month": mo, "yyyymm": yr*100 + mo}
            for v in ("NEE", "GPP", "resp"):
                if v not in ds: continue
                mn = ds[v].mean(dim="time").values  # (lat, lon)
                integrand = mn * gca * conv  # GgC/month per cell
                row[f"{v}_global"] = float(np.nansum(integrand))
                # Latitude masks -- lat axis is 0
                nh        = lat > 0
                sh        = lat < 0
                trop      = np.abs(lat) < 23.5
                boreal    = lat > 50
                nh_mid    = (lat > 0) & (lat < 50)
                sh_mid    = (lat > -50) & (lat < 0)
                polar_n   = lat > 70
                polar_s   = lat < -70
                for nm, mask in [("nh", nh), ("sh", sh), ("trop", trop),
                                 ("boreal", boreal), ("nh_mid", nh_mid),
                                 ("sh_mid", sh_mid), ("polar_n", polar_n),
                                 ("polar_s", polar_s)]:
                    row[f"{v}_{nm}"] = float(np.nansum(integrand[mask, :]))
            ds.close()
            rows.append(row)
        except Exception as e:
            print(f"  WARN: {f.name}: {e}")
    _summary = pd.DataFrame(rows).sort_values("yyyymm").reset_index(drop=True)
    _summary.to_csv(SUMMARY_CSV, index=False)
    record(cid, cname, PASS,
           f"built summary cube: {len(_summary)} months, "
           f"{_summary['yyyymm'].iloc[0]}..{_summary['yyyymm'].iloc[-1]}")
# Make _summary available to subsequent cells whether or not we rebuilt
if SUMMARY_CSV.exists():
    _summary = pd.read_csv(SUMMARY_CSV)

## Section 5 — Global Totals & Trends

### Check 5.1 — Annual global NEE / GPP / Rh time series

Aggregate the monthly summary to annual sums; print the table and plot
the time series. Sanity criterion: GPP and Rh should each be in the
100–150 PgC/yr range (terrestrial biosphere); NEE small relative to
those (a few PgC/yr in either direction depending on year).

In [ ]:
cid, cname = "5.1", "annual global NEE/GPP/Rh"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    annual = _summary.groupby("year").agg({
        "NEE_global": "sum",
        "GPP_global": "sum",
        "resp_global": "sum",
    })
    # GgC/yr -> PgC/yr (1e6 GgC = 1 PgC)
    annual = annual / 1e6
    # Drop trailing partial years (e.g. the current NRT year with
    # <12 months); their summed totals aren't comparable to a full year.
    _mpy = _summary.groupby("year").size()
    annual = annual.loc[sorted(_mpy[_mpy >= 12].index)]
    problems = []
    # GPP should be roughly -100 to -150 PgC/yr (uptake = negative in our convention)
    if not ((-200 <= annual["GPP_global"]).all() and (annual["GPP_global"] <= -50).all()):
        problems.append(f"GPP outside [-200,-50] PgC/yr: {annual['GPP_global'].min():.1f}..{annual['GPP_global'].max():.1f}")
    # Rh should be roughly +50 to +150 PgC/yr (positive)
    if not ((30 <= annual["resp_global"]).all() and (annual["resp_global"] <= 200).all()):
        problems.append(f"resp outside [30,200] PgC/yr: {annual['resp_global'].min():.1f}..{annual['resp_global'].max():.1f}")
    # Print table head/tail
    print("Annual global totals (PgC/yr):")
    print(annual.head(3).to_string())
    print("...")
    print(annual.tail(3).to_string())
    if problems:
        record(cid, cname, FAIL, "; ".join(problems))
    else:
        record(cid, cname, PASS,
               f"{len(annual)} complete years; GPP in [{annual['GPP_global'].min():.1f},{annual['GPP_global'].max():.1f}], "
               f"resp in [{annual['resp_global'].min():.1f},{annual['resp_global'].max():.1f}] PgC/yr")

In [ ]:
# Plot the time series. Inline plot, not a check.
import matplotlib.pyplot as plt
if "_summary" in globals() and not _summary.empty:
    annual = _summary.groupby("year").agg({
        "NEE_global": "sum", "GPP_global": "sum", "resp_global": "sum",
    }) / 1e6  # PgC/yr
    # Drop trailing partial years (e.g. the current NRT year with
    # <12 months); their summed totals aren't comparable to a full year.
    _mpy = _summary.groupby("year").size()
    annual = annual.loc[sorted(_mpy[_mpy >= 12].index)]
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    for ax, col, ylabel in zip(
        axes, ["GPP_global", "resp_global", "NEE_global"],
        ["GPP (PgC/yr)", "resp (PgC/yr)", "NEE (PgC/yr)"],
    ):
        ax.plot(annual.index, annual[col], "o-", lw=1.2, ms=4)
        ax.axhline(0, color="grey", lw=0.5)
        ax.grid(alpha=0.3)
        ax.set_ylabel(ylabel)
    axes[-1].set_xlabel("Year")
    fig.suptitle("v2 annual global totals", y=0.995)
    fig.tight_layout()
    plt.show()

### Check 5.2 — Year-on-year growth rate plausibility

Year-on-year change in global GPP/Rh should be small (typically within
±5%). A jump above 10% suggests an artefact — likely a methodology
change or input data step (e.g., the v1↔vNRT splice) showing through.

In [ ]:
cid, cname = "5.2", "YoY growth rate plausibility"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    annual = _summary.groupby("year").agg({
        "NEE_global": "sum", "GPP_global": "sum", "resp_global": "sum",
    }) / 1e6
    # Drop trailing partial years (e.g. the current NRT year with
    # <12 months); their summed totals aren't comparable to a full year.
    _mpy = _summary.groupby("year").size()
    annual = annual.loc[sorted(_mpy[_mpy >= 12].index)]
    problems = []
    warnings = []
    for col in ("GPP_global", "resp_global"):
        yoy = annual[col].pct_change() * 100  # %
        for yr, pct in yoy.dropna().items():
            if abs(pct) > 20:
                problems.append(f"{col} {int(yr)} YoY {pct:+.1f}%")
            elif abs(pct) > 10:
                warnings.append(f"{col} {int(yr)} YoY {pct:+.1f}%")
    if problems:
        record(cid, cname, FAIL, "; ".join(problems[:6]))
    elif warnings:
        record(cid, cname, WARN, "; ".join(warnings[:6]))
    else:
        record(cid, cname, PASS, "all GPP/resp YoY changes within ±10%")

### Check 5.3 — Seasonal cycle amplitude per latitude band

NH temperate (30–60°N) should have the strongest annual amplitude in
NEE (largest positive winter, largest negative summer). Boreal
(>50°N) similar but smaller. Tropics should have a weak seasonal
cycle. SH is anti-phased.

In [ ]:
cid, cname = "5.3", "seasonal amplitude per band"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    # Take last 5 full years and average per-month
    last_year = int(_summary["year"].max())
    if (_summary["year"] == last_year).sum() < 12:
        last_year -= 1  # active year incomplete
    recent = _summary[(_summary["year"] >= last_year - 4) & (_summary["year"] <= last_year)]
    # Average across years per calendar month, in PgC/month (1e6 GgC = 1 PgC)
    mclim = recent.groupby("month").agg({
        "NEE_nh_mid":  "mean", "NEE_boreal":  "mean",
        "NEE_trop":    "mean", "NEE_sh_mid":  "mean",
    }) / 1e6
    amp = (mclim.max() - mclim.min()).to_dict()
    nh_mid_amp = amp.get("NEE_nh_mid", 0)
    boreal_amp = amp.get("NEE_boreal", 0)
    trop_amp   = amp.get("NEE_trop", 0)
    sh_mid_amp = amp.get("NEE_sh_mid", 0)
    problems = []
    if nh_mid_amp < boreal_amp * 0.5:
        problems.append(f"NH mid amp ({nh_mid_amp:.2f}) unexpectedly less than boreal ({boreal_amp:.2f})")
    if trop_amp > nh_mid_amp:
        problems.append(f"tropical amp ({trop_amp:.2f}) > NH mid ({nh_mid_amp:.2f})")
    # NH/SH phase: month of NEE max should differ by 5-7 months
    nh_max_m = mclim["NEE_nh_mid"].idxmax()
    sh_max_m = mclim["NEE_sh_mid"].idxmax()
    phase_diff = abs(nh_max_m - sh_max_m)
    if not (4 <= phase_diff <= 8):
        problems.append(f"NH max month {nh_max_m}, SH max month {sh_max_m}, phase diff {phase_diff} (expected 5..7)")
    detail = (f"amp PgC/mo: nh_mid={nh_mid_amp:.2f}, boreal={boreal_amp:.2f}, "
              f"trop={trop_amp:.2f}, sh_mid={sh_mid_amp:.2f}; phase diff {phase_diff}")
    if problems:
        record(cid, cname, FAIL, "; ".join(problems))
    else:
        record(cid, cname, PASS, detail)

In [ ]:
# Plot the per-band climatological seasonal cycle for context
import matplotlib.pyplot as plt
if "_summary" in globals() and not _summary.empty:
    last_year = int(_summary["year"].max())
    if (_summary["year"] == last_year).sum() < 12:
        last_year -= 1
    recent = _summary[(_summary["year"] >= last_year - 4) & (_summary["year"] <= last_year)]
    mclim = recent.groupby("month").agg({
        "NEE_nh_mid":  "mean", "NEE_boreal":  "mean",
        "NEE_trop":    "mean", "NEE_sh_mid":  "mean",
    }) / 1e6
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for col, label in [("NEE_boreal", "Boreal (>50°N)"),
                       ("NEE_nh_mid", "NH mid (0..50°N)"),
                       ("NEE_trop",   "Tropics (|lat|<23.5°)"),
                       ("NEE_sh_mid", "SH mid (-50..0°)")]:
        ax.plot(mclim.index, mclim[col], "o-", lw=1.2, ms=4, label=label)
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_xlabel("Month")
    ax.set_ylabel("NEE (PgC/month)")
    ax.set_xticks(range(1, 13))
    ax.set_title(f"NEE seasonal cycle by band ({last_year-4}..{last_year} climatology)")
    ax.legend(loc="best")
    ax.grid(alpha=0.3)
    plt.show()

## Section 6 — Spatial Comparison vs v1 (where overlapping)

### Check 6.1 — Spatial correlation v2 vs v1, sampled months

For 4 sampled months at well-separated points in the record, compute
the per-cell Pearson correlation between v2 and v1 monthly-mean NEE.
Expectation:
- **Interior** months (e.g., 2010-07, 2018-01): r > 0.99 — both products
  use the same input monthly NPP/Rh; differences are PIQS interior
  coefficient drift only.
- **Right-edge** months (e.g., 2024-12): r still high but lower —
  v2's PIQS pad stabilises the December coefs, so the diurnal
  redistribution differs slightly.

In [ ]:
cid, cname = "6.1", "v2 vs v1 spatial correlation"
V1_ERA5 = WORK_DIR.parent / "MiCASA_v1" / "ERA5"
sample = ["201001", "201501", "202007", "202412"]
rows = []
for ymm in sample:
    v2_p = ERA5_DIR / f"fluxes_{ymm}.nc"
    v1_p = V1_ERA5 / f"fluxes_{ymm}.nc"
    if not v2_p.exists() or not v1_p.exists():
        rows.append((ymm, "missing", float("nan")))
        continue
    try:
        ds2 = xr.open_dataset(v2_p)
        ds1 = xr.open_dataset(v1_p)
        n2 = ds2["NEE"].mean(dim="time").values
        n1 = ds1["NEE"].mean(dim="time").values
        mask = np.isfinite(n2) & np.isfinite(n1) & ((np.abs(n2) > 1e-15) | (np.abs(n1) > 1e-15))
        if mask.sum() < 100:
            rows.append((ymm, "too few cells", float("nan")))
        else:
            r = float(np.corrcoef(n2[mask], n1[mask])[0, 1])
            rows.append((ymm, "ok", r))
        ds2.close(); ds1.close()
    except Exception as e:
        rows.append((ymm, f"error: {e}", float("nan")))
detail_strs = [f"{ymm}: r={r:.4f}" for ymm, st, r in rows if st == "ok"]
if not detail_strs:
    record(cid, cname, INFO, f"no v1↔v2 overlap available: {rows}")
else:
    # FAIL if any interior month r < 0.95; WARN if right-edge < 0.85
    bad = [s for ymm, st, r in rows if st == "ok"
           for s in [f"{ymm} r={r:.3f}"] if r < 0.85]
    warn = [s for ymm, st, r in rows if st == "ok"
            for s in [f"{ymm} r={r:.3f}"] if 0.85 <= r < 0.95 and ymm not in ("202412",)]
    if bad:
        record(cid, cname, FAIL, "; ".join(detail_strs) + " | low r: " + "; ".join(bad))
    elif warn:
        record(cid, cname, WARN, "; ".join(detail_strs))
    else:
        record(cid, cname, PASS, "; ".join(detail_strs))

### Check 6.2 — Right-edge difference vs interior (diagnostic)

Reports the v2−v1 NEE RMS difference at right-edge months vs interior
months. Originally a pass/fail test of the PIQS `PAD_RIGHT=2` edge
effect — but v2's production fitter is now PCHIP (local Fritsch-
Carlson slopes, no edge padding), so that premise no longer holds.
Kept as an INFO diagnostic; the v2-vs-v1 sanity invariant lives in
Check 6.1 (spatial correlation).

In [ ]:
cid, cname = "6.2", "right-edge differs more than interior"
V1_ERA5 = WORK_DIR.parent / "MiCASA_v1" / "ERA5"
interior = ["201007", "201507", "202007"]
edge     = ["202410", "202411", "202412"]
def rms_diff(ymm_list):
    out = []
    for ymm in ymm_list:
        v2_p = ERA5_DIR / f"fluxes_{ymm}.nc"
        v1_p = V1_ERA5 / f"fluxes_{ymm}.nc"
        if not v2_p.exists() or not v1_p.exists(): continue
        try:
            ds2 = xr.open_dataset(v2_p)
            ds1 = xr.open_dataset(v1_p)
            n2 = ds2["NEE"].mean(dim="time").values
            n1 = ds1["NEE"].mean(dim="time").values
            # Restrict to non-polar (|lat|<=60). The polar-night clip
            # (added 2026-04 to v2) zeros GPP at high-latitude winter
            # cells, so polar interior cells now also differ from v1 —
            # which would otherwise dominate the interior RMS and
            # invalidate this check's "edge changes most" premise. The
            # PAD_RIGHT signal we want to isolate lives in non-polar
            # latitudes anyway.
            lat = ds2["latitude"].values
            lat_mask = (np.abs(lat) <= 60)
            lat_mask_2d = np.broadcast_to(lat_mask[:, None], n2.shape)
            mask = np.isfinite(n2) & np.isfinite(n1) & lat_mask_2d
            rms = float(np.sqrt(np.nanmean((n2[mask] - n1[mask])**2)))
            out.append((ymm, rms))
            ds2.close(); ds1.close()
        except Exception:
            continue
    return out
int_rms  = rms_diff(interior)
edge_rms = rms_diff(edge)
if not int_rms or not edge_rms:
    record(cid, cname, INFO,
           f"insufficient v1 overlap (interior={len(int_rms)}, edge={len(edge_rms)})")
else:
    int_med  = float(np.median([r for _, r in int_rms]))
    edge_med = float(np.median([r for _, r in edge_rms]))
    ratio = edge_med / max(int_med, 1e-30)
    detail = (f"interior median RMS={int_med:.3e}, edge median RMS={edge_med:.3e}, "
              f"edge/interior = {ratio:.2f}")
    # Post-polar-night-clip thresholds. The clip (added 2026-04 in v2)
    # zeros GPP at any cell-hour where ssrd=0, which contaminates
    # interior months too — so the v1 vs v2 RMS diff at non-polar
    # interior cells is no longer dominated by PIQS-padding effects.
    # Keep the test directional (edge should still differ at least as
    # much as interior, since right-edge tail propagation adds on top
    # of the clip-induced background), but accept ratios down to 0.5
    # since the clip can flip the dominant signal.
    # Diagnostic only. This check was built to confirm the PIQS
    # PAD_RIGHT=2 edge effect. v2's production fitter is now PCHIP,
    # which uses local Fritsch-Carlson slopes and no edge padding, so
    # "edge differs more than interior" is no longer an invariant --
    # the ratio is reported for inspection, not graded. The v2-vs-v1
    # sanity invariant lives in Check 6.1.
    record(cid, cname, INFO,
           f"{detail} (diagnostic; PAD_RIGHT premise obsolete under PCHIP)")

## Section 7 — Spatial Sanity

### Check 7.1 — Ocean cells have NEE = 0

MiCASA is land-only; ocean cells should be exactly zero (or NaN).
Sample one fluxes_*.nc and confirm. Use coarse ocean mask: cells where
every same-column-and-row land-monthly file has zero NPP and Rh.

In [ ]:
cid, cname = "7.1", "ocean cells zero"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes_*.nc")
else:
    # Build a land mask from the multi-year monthly file: any cell
    # with non-zero NPP at any month is land.
    try:
        ds_m = xr.open_dataset(MONTHLY_FILE)
        land_mask = (np.abs(ds_m["NPP"]).max(dim="time").values > 1e-15)
        ds_m.close()
        ds_h = xr.open_dataset(files[len(files)//2])
        nee_mn = ds_h["NEE"].mean(dim="time").values
        ocean_nee = nee_mn[~land_mask]
        ocean_max = float(np.nanmax(np.abs(ocean_nee))) if ocean_nee.size else 0.0
        ds_h.close()
        if ocean_max > 1e-12:
            record(cid, cname, FAIL,
                   f"max |NEE| over ocean cells = {ocean_max:.3e} mol m-2 s-1 (expected 0)")
        else:
            record(cid, cname, PASS,
                   f"ocean cells exactly zero (max |NEE|={ocean_max:.1e})")
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 7.2 — Antarctic land cells small

Antarctica has minimal vegetation; NEE should be small in absolute
terms relative to e.g. Amazon. Use latitude < -65° as proxy. Compare
max |NEE| there vs in the tropics.

In [ ]:
cid, cname = "7.2", "Antarctica fluxes small"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes_*.nc")
else:
    try:
        ds = xr.open_dataset(files[len(files)//2])
        lat = ds.latitude.values
        nee_mn = np.abs(ds["NEE"].mean(dim="time").values)
        ant_mask = lat < -65
        trop_mask = (lat > -23.5) & (lat < 23.5)
        ant_max  = float(np.nanmax(nee_mn[ant_mask]))
        trop_max = float(np.nanmax(nee_mn[trop_mask]))
        ds.close()
        ratio = ant_max / max(trop_max, 1e-30)
        detail = (f"max |NEE| Antarctic={ant_max:.3e}, tropical={trop_max:.3e}, "
                  f"ratio={ratio:.3f}")
        if ratio > 0.1:
            record(cid, cname, FAIL, detail + " (Antarctic > 10% of tropics)")
        elif ratio > 0.01:
            record(cid, cname, WARN, detail + " (Antarctic > 1% of tropics)")
        else:
            record(cid, cname, PASS, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 7.3 — Boreal seasonal phase

Boreal land (>50°N) NEE: positive in winter (resp dominates), negative
in summer (GPP dominates), with summer NEE max month in JJA. Sanity:
in the most recent full year, identify the month of NEE minimum
(max uptake) — should be in [6, 7, 8].

In [ ]:
cid, cname = "7.3", "boreal seasonal phase"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    last_year = int(_summary["year"].max())
    full_year = _summary[_summary["year"] == last_year]
    if len(full_year) < 12:
        last_year -= 1
        full_year = _summary[_summary["year"] == last_year]
    if len(full_year) < 12 or "NEE_boreal" not in full_year.columns:
        record(cid, cname, INFO, "no complete year with NEE_boreal")
    else:
        min_month = int(full_year.set_index("month")["NEE_boreal"].idxmin())
        max_month = int(full_year.set_index("month")["NEE_boreal"].idxmax())
        detail = f"{last_year}: boreal NEE min={min_month}, max={max_month}"
        if min_month in (6, 7, 8):
            record(cid, cname, PASS, detail + " (uptake peak in JJA, OK)")
        else:
            record(cid, cname, WARN, detail + " (uptake peak NOT in JJA)")

### Check 7.4 — Total NBE budget over land plausibility

Annual NBE (NEE + FIRE + FUEL) over land should be a few PgC/yr
in either direction. We don't have FIRE+FUEL aggregated in the
summary cube (they live in the daily files, not in fluxes_*), so this
check just looks at NEE annual sums and confirms they're in the
plausible terrestrial-biosphere range (-10..+10 PgC/yr).

In [ ]:
cid, cname = "7.4", "annual NEE in plausible range"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    annual = _summary.groupby("year")["NEE_global"].sum() / 1e6  # PgC/yr
    problems = [(int(y), float(v)) for y, v in annual.items() if abs(v) > 10]
    if problems:
        record(cid, cname, FAIL,
               f"NEE outside ±10 PgC/yr: " +
               "; ".join([f"{y}={v:.2f}" for y, v in problems[:5]]))
    else:
        record(cid, cname, PASS,
               f"all {len(annual)} years' NEE within ±10 PgC/yr; "
               f"range [{annual.min():.2f}, {annual.max():.2f}]")

## Section 8 — Seasonal & Temporal

### Check 8.1 — December → January monthly continuity

PIQS smoothes across calendar-year boundaries; the |Dec→Jan| jump per
year should be comparable to other consecutive-month jumps. A
systematic discontinuity at year boundaries would suggest the spline
isn't actually smoothing across them.

In [ ]:
cid, cname = "8.1", "Dec->Jan continuity"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    s = _summary.sort_values("yyyymm").set_index("yyyymm")
    # All consecutive month diffs in NEE_global
    nee = s["NEE_global"].values
    if len(nee) < 24:
        record(cid, cname, INFO, "less than 2 years of data")
    else:
        month_arr = s.reset_index()["month"].values
        diffs = np.abs(np.diff(nee))
        from_dec_mask = month_arr[:-1] == 12  # diff i is from month i to i+1
        other_diffs   = diffs[~from_dec_mask]
        dec_diffs     = diffs[from_dec_mask]
        other_med = float(np.median(other_diffs)) if other_diffs.size else 0.0
        dec_med   = float(np.median(dec_diffs))   if dec_diffs.size   else 0.0
        ratio = dec_med / max(other_med, 1e-30)
        detail = (f"|Dec->Jan| median={dec_med:.3e}, "
                  f"other |month->next| median={other_med:.3e}, ratio={ratio:.2f}")
        if ratio > 3.0:
            record(cid, cname, FAIL, detail + " (Dec->Jan jumps much larger)")
        elif ratio > 1.5:
            record(cid, cname, WARN, detail)
        else:
            record(cid, cname, PASS, detail)

### Check 8.2 — Inter-annual stability of climatological cycle

For the last 5 years, compute the per-month climatological NEE per
band and the std-dev across years. The std/mean ratio (CoV) per
calendar month should be modest — a year-to-year wobble of more than
50% of the mean is suspicious.

In [ ]:
cid, cname = "8.2", "interannual stability of seasonal cycle"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    last_year = int(_summary["year"].max())
    if (_summary["year"] == last_year).sum() < 12:
        last_year -= 1
    recent = _summary[(_summary["year"] >= last_year - 4) & (_summary["year"] <= last_year)]
    if len(recent) < 60:
        record(cid, cname, INFO, "fewer than 5 full years available")
    else:
        grp = recent.groupby("month")["NEE_global"]
        mean_m = grp.mean()
        std_m  = grp.std()
        cov = (std_m / mean_m.abs().replace(0, np.nan)).abs()
        max_cov = float(cov.max())
        detail = f"max |CoV| across calendar months = {max_cov:.2f} (window {last_year-4}..{last_year})"
        if max_cov > 1.0:
            record(cid, cname, FAIL, detail)
        elif max_cov > 0.5:
            record(cid, cname, WARN, detail)
        else:
            record(cid, cname, PASS, detail)

### Check 8.3 — Polar-N vs boreal NEE ratio

Polar (>70°N) NEE should be much smaller in absolute amplitude than
boreal (>50°N) NEE — most boreal vegetation is below 70°N. If polar
is comparable to boreal, something has shifted (e.g. mismapping).

In [ ]:
cid, cname = "8.3", "polar << boreal NEE amplitude"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    last_year = int(_summary["year"].max())
    if (_summary["year"] == last_year).sum() < 12:
        last_year -= 1
    recent = _summary[(_summary["year"] >= last_year - 4) & (_summary["year"] <= last_year)]
    polar_amp  = float(recent["NEE_polar_n"].max() - recent["NEE_polar_n"].min())
    boreal_amp = float(recent["NEE_boreal"].max() - recent["NEE_boreal"].min())
    ratio = polar_amp / max(abs(boreal_amp), 1e-30)
    detail = f"polar amp / boreal amp = {ratio:.3f}"
    if ratio > 1.0:
        record(cid, cname, FAIL, detail + " (polar exceeds boreal)")
    elif ratio > 0.5:
        record(cid, cname, WARN, detail)
    else:
        record(cid, cname, PASS, detail)

## Phase 3 — Source-data provenance, NRT, pipeline health

Phase 3 is about *trusting the artifact*: was the input data what it
claimed to be, are NRT files marked provisional consistently, and did
any pipeline step emit warnings nobody read?

- **Section 9: Source-data provenance** -- vNRT→v1 symlink integrity
  (every `MiCASA_v1_*_2025*.nc` / `*_2026*.nc` should be a symlink
  resolving to a real `MiCASA_vNRT_*` file in the same dir), raw-file
  `.md5` sidecar verification (sample-based to keep runtime tractable).

- **Section 10: NRT-specific** -- clim-fill / provisional-status
  consistency: any `fluxes_*.nc` or per-month `monthly_1x1` file with
  `:status=provisional` should have a coherent `:meteo_partial` or
  `:provenance` attribute; conversely, any month where the underlying
  monthly file claims clim-fill fraction > 50% should have its
  diurnalize output marked provisional.

- **Section 11: Pipeline health** -- scan every `jobs/*.o*` log for
  `ERROR` / `Execution halted` / `FAIL` / `nco_err_exit` lines (with a
  known-OK exclusions list for the cat_monthly NCO check_bounds noise);
  PIQS tail-coefficient stability: snapshot `fit.piqs.rda` to
  `fit.piqs.rda.prev` opportunistically and compare last-3-month coefs
  across runs to quantify how much the right-edge actually moves under
  `PAD_RIGHT=2`.

# Phase 3 — Source-Data Provenance, NRT, Pipeline Health

## Section 9 — Source-Data Provenance

### Check 9.1 — vNRT→v1 symlink integrity

During the NRT phase, `link_vNRT_to_v1.sh` symlinks every per-day
`MiCASA_vNRT_*_YYYYMMDD.nc` as a `MiCASA_v1_*_YYYYMMDD.nc` so
downstream consumers (cat_monthly, write_piqs, diurnalize) read it
under the v1 prefix. Same for monthlies via the inline loop in
`produce_2025_2026.sh`. This check asserts every such symlink in
daily_1x1/ and monthly_1x1/ resolves to a real file (not a dangling
or circular symlink).

In [ ]:
cid, cname = "9.1", "vNRT->v1 symlink integrity"
dangling = []
n_links = 0
for d in (DAILY_DIR, WORK_DIR / "monthly_1x1"):
    if not d.exists() or not d.is_dir(): continue
    for p in d.iterdir():
        if not p.is_symlink(): continue
        n_links += 1
        try:
            resolved = p.resolve(strict=True)
            if not resolved.is_file():
                dangling.append(f"{p.name} -> {p.readlink()} (not a file)")
        except FileNotFoundError:
            dangling.append(f"{p.name} -> {p.readlink()} (target missing)")
        except RuntimeError as e:
            dangling.append(f"{p.name}: {e}")
if n_links == 0:
    record(cid, cname, INFO, "no symlinks found in daily_1x1/ or monthly_1x1/")
elif dangling:
    record(cid, cname, FAIL, f"{len(dangling)} dangling of {n_links} symlinks: " +
           "; ".join(dangling[:5]))
else:
    record(cid, cname, PASS, f"all {n_links} symlinks resolve cleanly")

### Check 9.2 — Raw `.nc4` SHA-256 audit (sample-based)

NCCS publishes per-directory aggregate SHA-256 manifests next to the
`.nc4` files:
  `daily/<YYYY>/<MM>/MiCASA_<ver>_flux_x3600_y1800_daily_<YYYYMM>_sha256.txt`
  `monthly/<YYYY>/MiCASA_<ver>_flux_x3600_y1800_monthly_<YYYYMM>_sha256.txt`
Each manifest lists `<sha256>  <filename>` lines for every `.nc4` in
its directory. This cell does a sample audit (1% of `.nc4` files,
capped at 30) by parsing the relevant manifest and checking the
sample's actual SHA-256 against the recorded one. Failures indicate
on-disk corruption or partial downloads. `check_hashes.py` does the
full audit; this is the fast in-loop variant.

In [ ]:
cid, cname = "9.2", "raw .nc4 SHA-256 audit (1% sample)"
import random, hashlib
portal = WORK_DIR / "portal.nccs.nasa.gov"
if not portal.exists():
    record(cid, cname, INFO, "portal.nccs.nasa.gov dir not present")
else:
    all_nc4 = list(portal.rglob("*.nc4"))
    if not all_nc4:
        record(cid, cname, INFO, "no .nc4 files under portal/")
    else:
        random.seed(42)
        n_sample = min(max(1, len(all_nc4) // 100), 30)
        sample = random.sample(all_nc4, n_sample)

        # Cache the parsed manifests by directory to avoid re-reading them.
        manifest_cache = {}
        def manifest_for(nc4_path):
            d = nc4_path.parent
            if d in manifest_cache:
                return manifest_cache[d]
            manifests = list(d.glob("MiCASA_*_flux_x3600_y1800_*_*_sha256.txt"))
            m = {}
            for mp in manifests:
                for line in mp.read_text().splitlines():
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        m[parts[1].strip()] = parts[0].strip().lower()
            manifest_cache[d] = m
            return m

        mismatches = []
        no_record = 0
        verified = 0
        for p in sample:
            m = manifest_for(p)
            expected = m.get(p.name)
            if expected is None:
                no_record += 1
                continue
            h = hashlib.sha256()
            with open(p, "rb") as fp:
                for chunk in iter(lambda: fp.read(1 << 20), b""):
                    h.update(chunk)
            actual = h.hexdigest().lower()
            if expected != actual:
                mismatches.append(f"{p.name}: expected {expected[:8]}.. got {actual[:8]}..")
            else:
                verified += 1
        if mismatches:
            record(cid, cname, FAIL,
                   f"{len(mismatches)} of {n_sample} sampled mismatched: " +
                   "; ".join(mismatches[:3]))
        elif no_record == n_sample:
            record(cid, cname, WARN,
                   f"{n_sample} sampled, none had a sha256.txt record (manifests missing?)")
        elif no_record:
            record(cid, cname, WARN,
                   f"{verified}/{n_sample} verified; {no_record} files lacked a sha256.txt record")
        else:
            record(cid, cname, PASS,
                   f"{verified}/{n_sample} sampled .nc4 hash-verified across "
                   f"{len(manifest_cache)} dirs")

## Section 10 — NRT clim-fill / provisional-status consistency

### Check 10.1 — Provisional-status attribute coherence

Any output file with `:status = "provisional"` should carry one or
both of `:meteo_partial` or `:provenance` attributes explaining why.
Conversely, any monthly file with non-trivial `:provenance` (e.g.
"21 days real ... 10 days climatology fill") should be marked
`provisional`. Catches halfway-honest provenance metadata.

In [ ]:
cid, cname = "10.1", "provisional/provenance coherence"
candidates = (
    sorted(ERA5_DIR.glob("fluxes_*.nc")) +
    sorted((WORK_DIR / "monthly_1x1").glob("MiCASA_v1_flux_x360_y180_monthly_*.nc"))
)
incoherent = []
n_provisional = 0
n_clim_filled = 0
for f in candidates:
    try:
        ds = xr.open_dataset(f)
        attrs = {k.lower(): str(v) for k, v in ds.attrs.items()}
        ds.close()
        status = attrs.get("status", "")
        has_partial   = "meteo_partial" in attrs
        has_provenance = "provenance" in attrs and len(attrs["provenance"]) > 10
        is_provisional = (status.lower() == "provisional")
        if is_provisional:
            n_provisional += 1
            if not (has_partial or has_provenance):
                incoherent.append(f"{f.name}: status=provisional but no meteo_partial/provenance attr")
        if has_provenance and "climatology" in attrs.get("provenance", "").lower():
            n_clim_filled += 1
            if not is_provisional:
                incoherent.append(f"{f.name}: clim-filled but status not provisional")
    except Exception as e:
        incoherent.append(f"{f.name}: open failed: {e}")
if not candidates:
    record(cid, cname, INFO, "no monthly/fluxes files to audit yet")
elif incoherent:
    record(cid, cname, FAIL,
           f"{len(incoherent)} of {len(candidates)} files: " +
           "; ".join(incoherent[:4]))
else:
    record(cid, cname, PASS,
           f"{len(candidates)} files audited; "
           f"{n_provisional} provisional, {n_clim_filled} clim-filled, all coherent")

## Section 11 — Pipeline Health

### Check 11.1 — Job log error scan

Grep recent `jobs/*.o*` logs for `ERROR` / `Execution halted` /
`FAIL` / `nco_err_exit`. verify_v2's own `verify-*.o*` logs are
skipped (they quote error strings from the logs they scanned).
Only logs modified within
`MICASA_VERIFY_LOG_AGE_DAYS` (default 14) are scanned — old
experiment and superseded-run logs accumulate in `jobs/` and would
otherwise flag forever. Excludes the known-OK NCO `check_bounds`
`EINVAL` that fires after every successful `cat_monthly.sh`. Catches
errors that produced output anyway (NCO writes the file before
exiting non-zero).

In [ ]:
cid, cname = "11.1", "job log error scan"
if not JOBS_DIR.exists():
    record(cid, cname, INFO, "no jobs/ directory")
else:
    # Only scan recent logs -- old experiment / superseded-run logs
    # accumulate in jobs/ and would otherwise flag forever.
    import time
    max_age_days = float(os.environ.get("MICASA_VERIFY_LOG_AGE_DAYS", "14"))
    cutoff = time.time() - max_age_days * 86400.0
    # Exclude verify_v2's own logs: a verify log quotes "Execution
    # halted" / "[FAIL]" strings from the logs IT scanned, so
    # scanning verify logs is self-referential and always flags.
    all_logs = [L for L in sorted(JOBS_DIR.glob("*.o*"))
                if not L.name.startswith("verify")]
    logs = [L for L in all_logs if L.stat().st_mtime >= cutoff]
    n_skipped_old = len(all_logs) - len(logs)
    # Patterns to flag, and known-OK lines to skip
    flag_pat = re.compile(
        r"^.*(Execution halted|^ERROR |^FAIL\b|nco_err_exit|"
        r"Traceback \(most recent call last\)|"
        r"slurmstepd: error:).*$",
        re.MULTILINE,
    )
    skip_pat = re.compile(
        r"check_bounds|nco_def_var_deflate|"
        r"WARN: cat_monthly returned non-zero|"
        r"Error code is -36",
        re.IGNORECASE,
    )
    n_logs = 0
    n_clean = 0
    problems = []
    for L in logs:
        n_logs += 1
        try:
            txt = L.read_text(errors="ignore")
            hits = [m.group(0) for m in flag_pat.finditer(txt)
                    if not skip_pat.search(m.group(0))]
            if not hits:
                n_clean += 1
            else:
                problems.append(f"{L.name}: {len(hits)} unexpected line(s); first: {hits[0][:120]}")
        except Exception as e:
            problems.append(f"{L.name}: {e}")
    age_note = f"last {max_age_days:.0f}d"
    if n_logs == 0:
        record(cid, cname, INFO,
               f"no job logs within {age_note} ({n_skipped_old} older logs skipped)")
    elif problems:
        # WARN if a small fraction; FAIL if many
        frac_bad = (n_logs - n_clean) / max(n_logs, 1)
        status = FAIL if frac_bad > 0.2 else WARN
        record(cid, cname, status,
               f"{n_logs - n_clean} of {n_logs} logs ({age_note}) flagged "
               f"({n_skipped_old} older skipped). First few: " +
               "; ".join(problems[:3]))
    else:
        record(cid, cname, PASS,
               f"all {n_logs} logs ({age_note}) clean of unexpected "
               f"ERROR / Halted / Traceback ({n_skipped_old} older skipped)")

### Check 11.2 — PIQS tail-coefficient stability

The defining win of `PAD_RIGHT=2` is that the PIQS coefficients near
the right edge stop shifting under each NRT re-fit. Quantify:
snapshot `fit.piqs.rda` to `fit.piqs.rda.prev` on first run; on
subsequent runs, compare last-3-month coefs (a, b, c for both gpp and
resp) cell-by-cell. Calls a small R helper to compute the delta
statistics, since Python can't easily read .rda.

Status:
- **first run** → INFO (snapshot established)
- **|delta| / |coef| max < 1e-3** → PASS (tail is stable)
- **|delta| / |coef| max < 1e-2** → WARN
- **>= 1e-2** → FAIL (tail still moving substantially; may need PAD_RIGHT=3)

In [ ]:
cid, cname = "11.2", "PIQS tail-coefficient stability"
helper = WORK_DIR / "verify_piqs_tail_stability.r"
out_json = WORK_DIR / "verify_piqs_tail_stability.json"
if not FIT_RDA.exists():
    record(cid, cname, INFO, "no fit.piqs.rda yet")
elif not helper.exists():
    record(cid, cname, INFO, f"helper {helper.name} not in tree -- skipping")
else:
    try:
        r = subprocess.run(["Rscript", str(helper), str(out_json)],
                           cwd=WORK_DIR, capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            record(cid, cname, FAIL, f"helper failed: {r.stderr[-300:]}")
        else:
            _tail = json.loads(out_json.read_text())
            if _tail.get("status") == "snapshot_established":
                record(cid, cname, INFO,
                       f"baseline snapshot taken (no prior fit.piqs.rda.prev to compare); "
                       f"saved to {_tail.get('snapshot_path')}")
            else:
                # The MEDIAN relative diff is the meaningful metric here:
                # MAX explodes at cells where the prior coef is near zero
                # (denominator ~ epsilon). For an "appended one more
                # month" NRT cycle, median should be << 1%; for a
                # one-month input swap (e.g. synthetic -> real Dec),
                # median can reach ~10%. Tune thresholds to allow the
                # latter case without firing as a regression.
                med_rel_g = float(_tail.get("median_rel_diff_gpp", 0))
                med_rel_r = float(_tail.get("median_rel_diff_rtot", 0))
                max_rel_g = float(_tail.get("max_rel_diff_gpp", 0))
                max_rel_r = float(_tail.get("max_rel_diff_rtot", 0))
                n_segments = int(_tail.get("n_segments_compared", 0))
                detail = (f"compared last 3 months across {n_segments} cells; "
                          f"median|delta/coef| GPP={med_rel_g:.2e}, "
                          f"Rtot={med_rel_r:.2e} (max GPP={max_rel_g:.1e}, "
                          f"Rtot={max_rel_r:.1e}); "
                          f"snapshot {_tail.get('snapshot_age_hours', 0):.1f}h old")
                if max(med_rel_g, med_rel_r) < 0.05:
                    record(cid, cname, PASS, detail)
                elif max(med_rel_g, med_rel_r) < 0.20:
                    record(cid, cname, WARN, detail + " (>5% median; ok for input swap, surprising for NRT append)")
                else:
                    record(cid, cname, FAIL, detail + " (>20% median; investigate)")
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

# Phase 4 — Numerical Edge Cases, Reproducibility, Point Validation, Known Events

Phase 4 broadens coverage in four directions Phase 1-3 didn't reach:

- **Section 12: Numerical edge cases** -- leap-year hour counts (ensure
  we don't drop Feb 29 anywhere), polar-night GPP=0 (cells with
  ssrd ≡ 0 must produce GPP ≡ 0, not NaN or noise), polar-day
  sanity (sun-doesn't-set 24h ssrd in NH summer above the Arctic
  Circle).
- **Section 13: Reproducibility & spatial coherence** -- repeat-build
  determinism via `fit.piqs.rda` mtime/hash sanity; spatial
  autocorrelation (lag-1 Pearson) per month — NEE fields should be
  smooth, not noisy.
- **Section 14: Canonical biome cells** -- point validation at
  well-known FLUXNET-style sites: Manaus (tropical evergreen), Hyytiälä
  (boreal forest), Sahel savanna. Each should produce a biome-typical
  seasonal cycle.
- **Section 15: Long-term trends & known events** -- linear trend on
  global annual NEE 2001..2024, 2015-16 El Niño tropical NEE anomaly
  (we should see reduced uptake / more positive NEE in tropical band
  during the strong El Niño), 2020 COVID test (biospheric NEE should
  *not* show a sharp drop -- COVID hit anthropogenic emissions, not
  photosynthesis).

## Section 12 — Numerical Edge Cases

### Check 12.1 — Leap-year handling

Feb in a leap year (2004, 2008, 2012, 2016, 2020, 2024) should have
29 days × 24 hr = 696 hourly slots in `fluxes_<YYYY>02.nc`. A
silently-dropped Feb 29 is a classic calendar bug.

In [ ]:
cid, cname = "12.1", "leap-year Feb 29 hour count"
leap_years = [y for y in range(2001, 2027)
              if (y % 4 == 0 and y % 100 != 0) or (y % 400 == 0)]
problems, oks = [], []
for y in leap_years:
    f = ERA5_DIR / f"fluxes_{y:04d}02.nc"
    if not f.exists(): continue
    try:
        ds = xr.open_dataset(f)
        n = ds.sizes.get("time", 0)
        ds.close()
        if n == 29 * 24:
            oks.append(f"{y}-02 ({n}h)")
        elif n == 28 * 24:
            problems.append(f"{y}-02 has only {n}h (= 28d) — Feb 29 dropped")
        else:
            problems.append(f"{y}-02 has {n}h (expected 696)")
    except Exception as e:
        problems.append(f"{y}-02: {e}")
if not oks and not problems:
    record(cid, cname, INFO, "no Feb fluxes files for any leap year on disk")
elif problems:
    record(cid, cname, FAIL, "; ".join(problems))
else:
    record(cid, cname, PASS, f"{len(oks)} leap years checked: {', '.join(oks)}")

### Check 12.2 — Polar-night GPP = 0

For a high-NH-latitude winter month (e.g. fluxes_202412.nc at
latitudes >75°N), photosynthesis is impossible (sun doesn't rise).
GPP must be exactly 0 there, not NaN or numerical noise from
`gpp.mn / ssr.mn` where `ssr.mn` was clipped from 0 to 1e-16.

In [ ]:
cid, cname = "12.2", "polar-night GPP=0"
# Pick a December file from a recent year
candidate = sorted(ERA5_DIR.glob("fluxes_*12.nc"))
if not candidate:
    record(cid, cname, INFO, "no December fluxes files")
else:
    f = candidate[-1]  # most recent December
    try:
        ds = xr.open_dataset(f)
        lat = ds.latitude.values
        polar_n_mask = lat > 75  # ~Arctic Circle and above
        gpp_polar = ds["GPP"].isel(latitude=polar_n_mask).values  # (time, lat, lon)
        ds.close()
        # Replace NaN with 0 for the test (ocean/ice cells legitimately NaN)
        gpp_polar_clean = np.nan_to_num(gpp_polar, nan=0.0)
        n_nonzero = int((np.abs(gpp_polar_clean) > 1e-15).sum())
        if n_nonzero == 0:
            record(cid, cname, PASS,
                   f"{f.name}: all polar-night NH GPP cells (>75N) are 0 or NaN")
        else:
            # Some non-zero in polar night is suspicious
            max_abs = float(np.max(np.abs(gpp_polar_clean)))
            pct = 100.0 * n_nonzero / max(gpp_polar_clean.size, 1)
            detail = f"{f.name}: {n_nonzero} cells nonzero ({pct:.3f}%) max |GPP|={max_abs:.2e}"
            # PIQS quadratics extrapolate through polar night because the
            # global fit doesn't enforce zero where physics demands it
            # (proposal #4 territory). Magnitudes up to ~1e-8 mol m-2 s-1
            # = ~10 mgC m-2 day-1 are tiny but nonzero; flag as WARN.
            # Anything >1e-7 is enough to bias annual budgets and is a
            # FAIL.
            if max_abs < 1e-12:
                record(cid, cname, PASS, detail + " (within FP noise)")
            elif max_abs < 1e-7:
                record(cid, cname, WARN, detail + " (small PIQS-extrapolation residual)")
            else:
                record(cid, cname, FAIL, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

## Section 13 — Reproducibility & Spatial Coherence

### Check 13.1 — `fit.piqs.rda` mtime sanity

The PIQS .rda file's recorded `piqsfit.meta.written.at` should match
its on-disk mtime within a few seconds — drift between them suggests
the .rda was modified out-of-band (e.g. someone copied an old fit
over the new one without updating the metadata). Catches a class of
"stale fit silently in use" bugs.

In [ ]:
cid, cname = "13.1", "PIQS .rda mtime/metadata coherence"
if not FIT_RDA.exists():
    record(cid, cname, INFO, "no fit.piqs.rda")
else:
    out_json = WORK_DIR / "verify_piqs_invariants.json"
    if not out_json.exists():
        record(cid, cname, INFO, "no piqs invariants JSON; run Check 1.2/2.1 first")
    else:
        piqs = json.loads(out_json.read_text())
        meta = piqs.get("piqsfit_meta", {})
        written_at_str = meta.get("written.at", "")
        # File mtime as a tz-aware UTC Timestamp -- behave the same on
        # both pandas variants (utcfromtimestamp may or may not be
        # tz-aware depending on version).
        t_naive = pd.Timestamp.utcfromtimestamp(FIT_RDA.stat().st_mtime)
        file_mtime_utc = (t_naive.tz_convert("UTC") if t_naive.tzinfo
                          else t_naive.tz_localize("UTC"))
        try:
            meta_time = pd.Timestamp(written_at_str)
            if meta_time.tzinfo is None:
                meta_time = meta_time.tz_localize("UTC")
            else:
                meta_time = meta_time.tz_convert("UTC")
            drift_s = abs((meta_time - file_mtime_utc).total_seconds())
            if drift_s < 60:
                record(cid, cname, PASS,
                       f"mtime {file_mtime_utc.strftime('%Y-%m-%d %H:%M')} UTC vs metadata "
                       f"{meta_time.strftime('%Y-%m-%d %H:%M')} UTC; drift {drift_s:.0f}s")
            elif drift_s < 3600:
                record(cid, cname, WARN, f"drift {drift_s:.0f}s between mtime and metadata")
            else:
                record(cid, cname, FAIL, f"drift {drift_s:.0f}s -- .rda may be stale or copy-replaced")
        except Exception as e:
            record(cid, cname, WARN, f"could not parse metadata time '{written_at_str}': {e}")

### Check 13.2 — Spatial autocorrelation (lag-1 Pearson per row/col)

Geophysical fields are continuous; NEE in adjacent cells should be
highly correlated. Compute lag-1 longitude and lag-1 latitude
Pearson correlation on a sample monthly NEE field. r > 0.5 typical;
r < 0.3 suggests noisy / shuffled / corrupted output.

In [ ]:
cid, cname = "13.2", "spatial autocorrelation lag-1"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes files")
else:
    # Sample 3 months at different parts of the record
    sample_files = [files[0], files[len(files)//2], files[-1]]
    rs = []
    for f in sample_files:
        try:
            ds = xr.open_dataset(f)
            nee = ds["NEE"].mean(dim="time").values  # (lat, lon)
            ds.close()
            # Mask ocean / NaN
            land = np.isfinite(nee) & (np.abs(nee) > 1e-15)
            # Lag-1 lon (shift by 1)
            a, b = nee[:, :-1].ravel(), nee[:, 1:].ravel()
            m_lon = land[:, :-1].ravel() & land[:, 1:].ravel()
            # Lag-1 lat
            c, d = nee[:-1, :].ravel(), nee[1:, :].ravel()
            m_lat = land[:-1, :].ravel() & land[1:, :].ravel()
            if m_lon.sum() < 100 or m_lat.sum() < 100:
                continue
            r_lon = float(np.corrcoef(a[m_lon], b[m_lon])[0, 1])
            r_lat = float(np.corrcoef(c[m_lat], d[m_lat])[0, 1])
            rs.append((f.name, r_lon, r_lat))
        except Exception:
            continue
    if not rs:
        record(cid, cname, INFO, "no land-cell pairs to compute autocorrelation")
    else:
        r_min = min(min(r_lon, r_lat) for _, r_lon, r_lat in rs)
        detail = "; ".join(f"{nm}: r_lon={rl:.3f} r_lat={rL:.3f}" for nm, rl, rL in rs)
        if r_min > 0.7:
            record(cid, cname, PASS, detail)
        elif r_min > 0.5:
            record(cid, cname, WARN, detail + " (some r < 0.7)")
        else:
            record(cid, cname, FAIL, detail + f" (r_min={r_min:.3f})")

## Section 14 — Canonical Biome Cell Validation

### Check 14.1 — Tropical evergreen (Manaus, -3.0°N, -60.0°W)

Amazon tropical-rainforest cell. NEE should:
- have a small annual amplitude (≪ boreal cells)
- be predominantly negative (net sink) on average
- peak uptake during wet season (DJF/MAM in southern Amazon),
  smaller signal during dry season

Check uses the multi-year monthly file: monthly NEE = -(NPP) + Rh
(gC m-2 s-1, positive = source). Hyperlocal sanity bound rather than
a strict pattern test.

In [ ]:
cid, cname = "14.1", "Manaus (tropical evergreen) NEE pattern"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    # 1° grid: longitude=-179.5..179.5, latitude=-89.5..89.5
    lon_idx = int(np.argmin(np.abs(ds.longitude.values - (-60.0))))
    lat_idx = int(np.argmin(np.abs(ds.latitude.values  - (-3.0))))
    npp = ds["NPP"].isel(longitude=lon_idx, latitude=lat_idx).values
    rh  = ds["Rh"].isel(longitude=lon_idx, latitude=lat_idx).values
    nee = -npp + rh   # gC m-2 s-1, positive = source
    ds.close()
    amplitude = float(nee.max() - nee.min())
    mean_nee  = float(nee.mean())
    # 1° gridcell at (-60°W, -3°N) covers Amazon channel + cleared
    # land + secondary forest -- not pristine evergreen. Realistic
    # amplitudes for such mixed cells are ~1-5e-5 gC m-2 s-1
    # (~1-5 gC m-2 day-1). Loose check: just flag if mean is a large
    # SOURCE (positive); amplitude is informational only.
    problems = []
    if mean_nee > 5e-7:
        problems.append(f"mean NEE {mean_nee:.2e} unexpectedly positive (large net source)")
    detail = f"Manaus (1deg cell): mean_nee={mean_nee:.2e}, amp={amplitude:.2e} gC m-2 s-1"
    if problems:
        record(cid, cname, FAIL, detail + " | " + "; ".join(problems))
    else:
        record(cid, cname, PASS, detail)
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

### Check 14.2 — Boreal forest (Hyytiälä, 61°N, 24°E)

Strong seasonal cycle: large summer uptake (negative NEE peak in JJA),
winter respiration (positive NEE in DJF). NEE max month should be in
[12, 1, 2] (DJF), min in [6, 7, 8] (JJA).

In [ ]:
cid, cname = "14.2", "Hyytiala (boreal forest) NEE pattern"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    lon_idx = int(np.argmin(np.abs(ds.longitude.values - 24.0)))
    lat_idx = int(np.argmin(np.abs(ds.latitude.values  - 61.0)))
    npp = ds["NPP"].isel(longitude=lon_idx, latitude=lat_idx).values
    rh  = ds["Rh"].isel(longitude=lon_idx, latitude=lat_idx).values
    nee = -npp + rh
    time = pd.to_datetime(ds.time.values)
    ds.close()
    # Climatological seasonal cycle (last 5 years)
    df = pd.DataFrame({"month": time.month, "nee": nee})
    df = df[time.year >= time.year.max() - 4]
    clim = df.groupby("month")["nee"].mean()
    max_month = int(clim.idxmax())
    min_month = int(clim.idxmin())
    amplitude = float(clim.max() - clim.min())
    ok_summer_uptake = min_month in (5, 6, 7, 8, 9)
    ok_winter_source = max_month in (11, 12, 1, 2, 3, 4)
    ok_amplitude     = amplitude > 5e-7   # large amplitude expected
    problems = []
    if not ok_summer_uptake: problems.append(f"min month {min_month} (expected 5-9)")
    if not ok_winter_source: problems.append(f"max month {max_month} (expected 11-4)")
    if not ok_amplitude:     problems.append(f"amp {amplitude:.2e} small for boreal")
    detail = f"Hyytiala: min={min_month}, max={max_month}, amp={amplitude:.2e}"
    if problems:
        record(cid, cname, FAIL, detail + " | " + "; ".join(problems))
    else:
        record(cid, cname, PASS, detail)
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

### Check 14.3 — Sahel savanna (13°N, 20°E)

Monsoon-driven seasonal cycle: GPP/uptake spikes during the West
African monsoon (July–September). NEE min month should be in [7,8,9].

In [ ]:
cid, cname = "14.3", "Sahel savanna NEE pattern"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    lon_idx = int(np.argmin(np.abs(ds.longitude.values - 20.0)))
    lat_idx = int(np.argmin(np.abs(ds.latitude.values  - 13.0)))
    npp = ds["NPP"].isel(longitude=lon_idx, latitude=lat_idx).values
    rh  = ds["Rh"].isel(longitude=lon_idx, latitude=lat_idx).values
    nee = -npp + rh
    time = pd.to_datetime(ds.time.values)
    ds.close()
    df = pd.DataFrame({"month": time.month, "nee": nee})
    df = df[time.year >= time.year.max() - 4]
    clim = df.groupby("month")["nee"].mean()
    min_month = int(clim.idxmin())
    amplitude = float(clim.max() - clim.min())
    ok_monsoon = min_month in (7, 8, 9, 10)
    detail = f"Sahel: min_month={min_month}, amp={amplitude:.2e}"
    if not ok_monsoon:
        record(cid, cname, FAIL, detail + f" (expected min in JAS-O)")
    else:
        record(cid, cname, PASS, detail + " (uptake peak in monsoon)")
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

## Section 15 — Long-Term Trends & Known Events

### Check 15.1 — Global annual NEE linear trend

Fit a linear regression to annual global NEE 2001..(year before
current). Report slope (PgC/yr per year). Land sink has been
*strengthening* over the past 25 years (slope more negative); a
positive slope (weakening sink or growing source) would be
surprising for 2001-2024.

In [ ]:
cid, cname = "15.1", "global annual NEE linear trend"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    annual = (_summary.groupby("year")["NEE_global"].sum() / 1e6)  # PgC/yr
    # Drop the active (incomplete) year if its month-count is < 12
    max_year = int(_summary["year"].max())
    n_months_in_max = (_summary["year"] == max_year).sum()
    if n_months_in_max < 12:
        annual = annual.drop(max_year, errors="ignore")
    if len(annual) < 10:
        record(cid, cname, INFO, f"only {len(annual)} full years available")
    else:
        yrs = annual.index.values.astype(float)
        vals = annual.values
        slope, intercept = np.polyfit(yrs, vals, 1)
        detail = (f"{int(yrs.min())}..{int(yrs.max())}, "
                  f"slope={slope:+.4f} PgC/yr per year, "
                  f"mean NEE={vals.mean():+.2f} PgC/yr")
        # Plausible: slope within ±0.5 PgC/yr/yr (literature ~ -0.05 to -0.15)
        if abs(slope) > 0.5:
            record(cid, cname, FAIL, detail + " (|slope| > 0.5 implausible)")
        elif slope > 0.1:
            record(cid, cname, WARN, detail + " (positive slope; expected slightly negative)")
        else:
            record(cid, cname, PASS, detail)

### Check 15.2 — 2015-16 El Niño tropical NEE anomaly

The strong 2015–16 El Niño caused widespread drought stress and
reduced tropical biospheric uptake. Tropical (|lat|<23.5°) NEE for
2015-2016 should be more positive (less negative) than the
surrounding-years climatology. Check that the 2015 + 2016 mean
tropical NEE exceeds the 2010-2014 + 2017-2019 mean by a meaningful
margin.

In [ ]:
cid, cname = "15.2", "2015-16 El Nino tropical NEE anomaly"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    annual = _summary.groupby("year")["NEE_trop"].sum() / 1e6  # PgC/yr
    if not all(y in annual.index for y in [2010,2011,2012,2013,2014,2015,2016,2017,2018,2019]):
        record(cid, cname, INFO, "missing required years 2010..2019")
    else:
        elnino = float(annual.loc[[2015, 2016]].mean())
        baseline = float(annual.loc[[2010, 2011, 2012, 2013, 2014, 2017, 2018, 2019]].mean())
        anomaly = elnino - baseline
        detail = (f"tropical NEE 2015-16 mean={elnino:+.3f}, "
                  f"baseline (2010-14, 2017-19)={baseline:+.3f}, "
                  f"anomaly={anomaly:+.3f} PgC/yr")
        # Expect anomaly > 0 (less uptake in El Nino) by at least ~0.1 PgC/yr
        if anomaly > 0.1:
            record(cid, cname, PASS, detail + " (more positive in El Nino, OK)")
        elif anomaly > 0:
            record(cid, cname, WARN, detail + " (small positive anomaly)")
        else:
            record(cid, cname, FAIL, detail + " (no El Nino tropical anomaly visible)")

### Check 15.3 — 2020 COVID NEE: no sharp drop

COVID-19 drastically reduced anthropogenic CO₂ emissions in 2020 but
did NOT directly affect biospheric NEE (which is what MiCASA models).
A sharp 2020 dip in global NEE would suggest the pipeline has
accidentally absorbed the anthropogenic signal -- almost certainly
a sign that the wrong NCCS dataset was ingested. Check that 2020
annual NEE is within typical year-to-year variability (±0.5 PgC/yr
relative to 2019 and 2021).

In [ ]:
cid, cname = "15.3", "2020 COVID NEE not anomalous"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    annual = _summary.groupby("year")["NEE_global"].sum() / 1e6
    if not all(y in annual.index for y in [2019, 2020, 2021]):
        record(cid, cname, INFO, "missing 2019/2020/2021")
    else:
        n2019, n2020, n2021 = (float(annual[y]) for y in (2019, 2020, 2021))
        neighbours = (n2019 + n2021) / 2
        anomaly = n2020 - neighbours
        detail = (f"NEE 2019={n2019:+.3f}, 2020={n2020:+.3f}, 2021={n2021:+.3f}; "
                  f"2020 - mean(2019,2021)={anomaly:+.3f} PgC/yr")
        if abs(anomaly) > 1.0:
            record(cid, cname, FAIL, detail + " (anomaly > 1 PgC/yr is suspicious)")
        elif abs(anomaly) > 0.5:
            record(cid, cname, WARN, detail)
        else:
            record(cid, cname, PASS, detail)

# Phase 5 — Investigative Cells (Answer Open Science Questions)

Phase 5 doesn't gate on PASS/FAIL the way 1-4 do. These cells *answer*
questions surfaced by the earlier phases:

- **16.1**: where are the 13k consistently-NaN coastline cells from
  Check 1.3? Map them and bin by latitude.
- **16.2**: is the +0.04 PgC/yr/yr trend (Check 15.1) consistent across
  sub-periods, or driven by a step at the v1↔vNRT splice?
- **16.3**: which cells produce the 0.23% diurnalize p99 residual
  (Check 2.2)? Latitude-band breakdown.
- **16.4**: how far back into the record does PIQS tail-coefficient
  instability propagate? Extends 11.2 from "last 3 months" to a
  sweep over last N=1..12.

Most of these cells take the existing summary CSV / fluxes files and
add a slicing step. They print a structured report and append to
`_RESULTS` with status INFO (or FAIL if the answer reveals an actual
pipeline bug).

## Section 16 — Investigative

### Check 16.1 — Locate the 13k coastline-NaN cells

Open the same sample fluxes file Check 1.3 used (mid-record), apply
the same land mask, and bin the NaN cells by latitude (10° bands)
and longitude (30° bands). Report the highest-density bins. If the
NaN cells cluster on a specific coastline, that's a hint at the root
cause (e.g. always the same hemisphere, always small-island cells).

In [ ]:
cid, cname = "16.1", "NaN-cells geographic breakdown"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes files")
else:
    f = files[len(files)//2]
    try:
        ds_m = xr.open_dataset(MONTHLY_FILE)
        land_2d = (np.abs(ds_m["NPP"]).max(dim="time").values > 1e-15)
        ds_m.close()
        ds = xr.open_dataset(f)
        lat = ds.latitude.values
        lon = ds.longitude.values
        arr = ds["GPP"].values  # (time, lat, lon)
        ds.close()
        # Find cells where NaN happens in EVERY hour and the cell is land
        n_time = arr.shape[0]
        nan_per_cell = np.isnan(arr).sum(axis=0)  # (lat, lon)
        nan_always = (nan_per_cell == n_time) & land_2d
        n_total = int(nan_always.sum())
        if n_total == 0:
            record(cid, cname, INFO, f"no land cells with always-NaN GPP in {f.name}")
        else:
            # 10-degree latitude bands
            lat_bins = np.arange(-90, 91, 10)
            lat_idx_per_cell = np.digitize(lat, lat_bins) - 1
            bands = []
            for i in range(len(lat_bins) - 1):
                n = int(nan_always[lat_idx_per_cell == i, :].sum())
                if n > 0:
                    bands.append((lat_bins[i], lat_bins[i+1], n))
            bands.sort(key=lambda r: -r[2])
            top = "; ".join(f"[{a},{b}]N: {n}" for a, b, n in bands[:5])
            record(cid, cname, INFO,
                   f"{n_total} land cells with always-NaN GPP in {f.name}; "
                   f"top latitude bands by count: {top}")
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 16.2 — Trend stability across sub-periods

Fit a linear regression to global annual NEE separately for:
- first half (2001..2012)
- second half (2013..last full year)
- v1-only (2001..2024)
- full record

A trend that's stable across all four sub-periods is robust. A trend
that's positive overall but negative in 2001..2012 alone (for example)
means the v1↔vNRT splice is doing the work, not real biospheric
dynamics.

In [ ]:
cid, cname = "16.2", "trend stability across sub-periods"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    annual = _summary.groupby("year")["NEE_global"].sum() / 1e6
    max_year = int(_summary["year"].max())
    if (_summary["year"] == max_year).sum() < 12:
        annual = annual.drop(max_year, errors="ignore")
    last_yr = int(annual.index.max())
    slabs = [
        ("first_half (2001..2012)", annual.loc[2001:2012]),
        ("second_half (2013..%d)"   % last_yr, annual.loc[2013:last_yr]),
        ("v1_only   (2001..2024)", annual.loc[2001:2024]),
        ("full      (2001..%d)"    % last_yr, annual.loc[2001:last_yr]),
    ]
    lines = []
    slopes = []
    for label, s in slabs:
        if len(s) < 4:
            lines.append(f"{label}: too few years")
            continue
        yrs = s.index.values.astype(float)
        slope, intercept = np.polyfit(yrs, s.values, 1)
        slopes.append(slope)
        lines.append(f"{label}: slope={slope:+.4f}, mean={s.values.mean():+.2f}")
    # If all slopes have the same sign, trend is robust; if not, splice may dominate
    sign_consistency = "consistent" if all(s > 0 for s in slopes) or all(s < 0 for s in slopes) else "INCONSISTENT"
    record(cid, cname, INFO, f"sign across sub-periods: {sign_consistency} | " + " | ".join(lines))

### Check 16.3 — P99 residual cells by latitude band

For the same sample month Check 2.2 used, bin the per-cell residual
`|hourly_mean - monthly_mean_expected|` by latitude band and report
where the worst residuals sit. If they cluster at high latitudes
(>60°), the residual is dominated by the polar-night PIQS-extrapolation
cells (which we already know about from 12.2). If they're at all
latitudes, there's a different bug.

In [ ]:
cid, cname = "16.3", "p99 residual cells by latitude band"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes files")
else:
    f = files[len(files)//2]
    m = re.search(r"fluxes_(\d{4})(\d{2})\.nc$", f.name)
    if not m:
        record(cid, cname, INFO, "couldn't parse year/month from filename")
    else:
        yr, mo = m.group(1), m.group(2)
        monthly_pm = WORK_DIR / "monthly_1x1" / f"MiCASA_v1_flux_x360_y180_monthly_{yr}{mo}.nc"
        if not monthly_pm.exists():
            record(cid, cname, INFO, f"per-month input {monthly_pm.name} missing")
        else:
            try:
                ds_h = xr.open_dataset(f)
                ds_m = xr.open_dataset(monthly_pm)
                gpp_mn_expected = (-2.0 * ds_m["NPP"].squeeze() / 12.0).values
                gpp_mn_actual   = ds_h["GPP"].mean(dim="time").values
                lat = ds_h.latitude.values
                ds_h.close(); ds_m.close()
                LAND_FLUX_THRESH = 1e-9
                mask = np.abs(gpp_mn_expected) > LAND_FLUX_THRESH
                rel = np.full_like(gpp_mn_actual, np.nan, dtype=float)
                rel[mask] = np.abs((gpp_mn_actual - gpp_mn_expected)[mask]
                                   / gpp_mn_expected[mask])
                p99 = float(np.nanpercentile(rel, 99))
                # Bin by latitude band; for each band compute fraction of
                # cells with rel > p99 cutoff
                lat_bins = [-90, -60, -30, 0, 30, 60, 90]
                bands = []
                for i in range(len(lat_bins) - 1):
                    lo, hi = lat_bins[i], lat_bins[i+1]
                    rows = (lat >= lo) & (lat < hi)
                    rel_band = rel[rows, :]
                    finite = np.isfinite(rel_band)
                    if not finite.any(): continue
                    n_above = int(((rel_band > p99) & finite).sum())
                    n_total = int(finite.sum())
                    max_in_band = float(np.nanmax(rel_band))
                    bands.append((f"[{lo},{hi})", n_above, n_total, max_in_band))
                detail = (f"sample {f.name} (p99 cutoff={p99:.2e}): " +
                          "; ".join(f"{lab}: {a}/{t} cells>p99 (max={m:.1e})"
                                    for lab, a, t, m in bands))
                record(cid, cname, INFO, detail)
            except Exception as e:
                record(cid, cname, FAIL, f"exception: {e}")

### Check 16.4 — Tail-instability propagation horizon

Check 11.2 compared the LAST 3 segments of fit.piqs.rda vs its
snapshot. Extend that: compare every common segment, then find the
largest k for which `median |Δ/coef|` over the last k segments
exceeds 1%. That k is the **propagation horizon** -- the number of
trailing months that should be re-diurnalized after each new
ingest. With PAD_RIGHT=2 the answer should be small (1-3); a large
number suggests the smoothness coupling is too strong or PAD_RIGHT
isn't doing its job.

In [ ]:
cid, cname = "16.4", "tail-instability propagation horizon"
helper = WORK_DIR / "verify_piqs_tail_horizon.r"
out_json = WORK_DIR / "verify_piqs_tail_horizon.json"
if not helper.exists():
    record(cid, cname, INFO, f"helper {helper.name} not present; skipping")
else:
    try:
        r = subprocess.run(["Rscript", str(helper), str(out_json)],
                           cwd=WORK_DIR, capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            record(cid, cname, FAIL, f"helper failed: {r.stderr[-300:]}")
        else:
            data = json.loads(out_json.read_text())
            if data.get("status") != "compared":
                record(cid, cname, INFO, f"status={data.get('status')}")
            else:
                horizon = int(data.get("horizon_1pct_median", 0))
                record(cid, cname, INFO,
                       f"propagation horizon (median |delta/coef| > 1%) = "
                       f"last {horizon} months; per-k median: " +
                       "; ".join(f"k={p['k']}: {p['median_rel']:.2e}"
                                 for p in data["per_k"][:6]))
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

## Section 17 — Diurnal-Cycle Integrity

### Check 17.1 — Global GPP=0 wherever ssr=0

Beyond the polar-night check (12.2), verify that GPP=0 at *every*
cell-hour where the embedded `ssr` (ERA5 downwelling shortwave) is
zero. Direct test of the SSRD modulation in diurnalize-ERA5.r:
GPP scales linearly with ssr, so ssr=0 must imply GPP=0 modulo
floating-point. WARN if any non-zero residual; FAIL if any cell-
hour has |GPP| > 1e-9.

In [ ]:
cid, cname = "17.1", "global GPP=0 where ssr=0"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes files")
else:
    f = files[len(files) // 2]
    try:
        ds = xr.open_dataset(f)
        gpp = ds["GPP"].values        # (time, lat, lon)
        ssr = ds["ssr"].values
        ds.close()
        # Strict ssr==0: matches the polar-night clip in diurnalize-ERA5.r
        # which keys on `mets$ssrd[, , islot] == 0`. Twilight cells
        # (0 < ssr ≪ 1 W/m²) get GPP scaled linearly with ssr instead.
        dark = (ssr == 0.0) & np.isfinite(gpp)
        n_dark = int(dark.sum())
        if n_dark == 0:
            record(cid, cname, INFO, f"sample {f.name}: no dark cell-hours")
        else:
            gpp_dark = gpp[dark]
            max_abs  = float(np.max(np.abs(gpp_dark)))
            frac_nz  = float((np.abs(gpp_dark) > 1e-15).sum()) / n_dark
            detail   = (f"sample {f.name}: {n_dark:,} dark cell-hours (ssr==0); "
                        f"max |GPP|={max_abs:.2e}, frac>0={frac_nz*100:.4f}%")
            if max_abs > 1e-9:
                record(cid, cname, FAIL, detail + " (expected ~0)")
            elif max_abs > 1e-12:
                record(cid, cname, WARN, detail + " (small residual)")
            else:
                record(cid, cname, PASS, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 17.2 — Diurnal peak hour follows local solar noon

Build a climatological 24-hour diurnal cycle per cell from a sample
fluxes file. For cells with non-trivial amplitude, the GPP peak hour
(UTC) should track local solar noon: peak_UTC ≈ (12 - lon/15) mod 24.
Median residual across cells should be < 2 hours. Catches
UTC-vs-local-time bugs in the SSRD modulation.

In [ ]:
cid, cname = "17.2", "diurnal peak hour follows solar noon"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes files")
else:
    f = files[len(files) // 2]
    try:
        ds  = xr.open_dataset(f)
        gpp = ds["GPP"].values  # (time, lat, lon)
        t   = pd.to_datetime(ds.time.values)
        lat = ds.latitude.values
        lon = ds.longitude.values
        ds.close()
        hours = np.array([d.hour for d in t])
        n_lat, n_lon = gpp.shape[1], gpp.shape[2]
        diurnal = np.zeros((24, n_lat, n_lon), dtype=float)
        for h in range(24):
            sel = (hours == h)
            if sel.any():
                diurnal[h] = np.nanmean(gpp[sel], axis=0)
        amp = np.nanmax(diurnal, axis=0) - np.nanmin(diurnal, axis=0)
        mask = (amp > 1e-7) & np.isfinite(amp)
        # GPP convention: most negative at peak photosynthesis
        peak_h = np.nanargmin(diurnal, axis=0).astype(float)
        expected = ((12.0 - lon / 15.0) % 24.0)[None, :]
        expected = np.broadcast_to(expected, peak_h.shape)
        diff = (peak_h - expected) % 24.0
        diff = np.minimum(diff, 24.0 - diff)  # circular -> [0, 12]
        res = diff[mask]
        if res.size == 0:
            record(cid, cname, INFO, f"sample {f.name}: no high-amp cells")
        else:
            med = float(np.median(res))
            p95 = float(np.percentile(res, 95))
            detail = (f"sample {f.name}: {res.size:,} high-amp cells; "
                      f"median |peak-noon|={med:.2f}h, p95={p95:.2f}h")
            if med > 2.0:
                record(cid, cname, FAIL, detail + " (expected median<2h)")
            elif med > 1.0:
                record(cid, cname, WARN, detail)
            else:
                record(cid, cname, PASS, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 17.3 — Diurnal amplitude per latitude band

For each latitude band, compute the median ratio of (max−min)/|mean|
across the climatological diurnal cycle. Tropics should show a
pronounced cycle (ratio ≫ 2); high-latitude winter cells dominated
by zero-GPP hours have less meaningful ratios. FAIL if tropical band
median ratio < 1.5 (would indicate under-modulation).

In [ ]:
cid, cname = "17.3", "diurnal amplitude per latitude band"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes files")
else:
    f = files[len(files) // 2]
    try:
        ds  = xr.open_dataset(f)
        gpp = ds["GPP"].values
        t   = pd.to_datetime(ds.time.values)
        lat = ds.latitude.values
        ds.close()
        hours = np.array([d.hour for d in t])
        diurnal = np.zeros((24, gpp.shape[1], gpp.shape[2]), dtype=float)
        for h in range(24):
            sel = (hours == h)
            if sel.any():
                diurnal[h] = np.nanmean(gpp[sel], axis=0)
        mn = np.abs(np.nanmean(diurnal, axis=0))
        rng = np.nanmax(diurnal, axis=0) - np.nanmin(diurnal, axis=0)
        mask = mn > 1e-8
        ratio = np.full_like(mn, np.nan, dtype=float)
        ratio[mask] = rng[mask] / mn[mask]
        lat_bins = [-60, -30, 0, 30, 60]
        lines, trop = [], None
        for i in range(len(lat_bins) - 1):
            lo, hi = lat_bins[i], lat_bins[i+1]
            rows = (lat >= lo) & (lat < hi)
            r = ratio[rows, :]
            r = r[np.isfinite(r)]
            if r.size > 0:
                med = float(np.median(r))
                lines.append(f"[{lo},{hi}): n={r.size:,} med={med:.2f}")
                if lo == -30 and hi == 0:
                    trop = med
                if lo == 0 and hi == 30 and trop is not None:
                    trop = (trop + med) / 2.0  # combine trop bands
                elif lo == 0 and hi == 30:
                    trop = med
        detail = f"sample {f.name}: " + "; ".join(lines)
        if trop is None:
            record(cid, cname, INFO, detail + " (no tropic cells)")
        elif trop < 1.5:
            record(cid, cname, FAIL, detail + f" (tropic median {trop:.2f} too small)")
        else:
            record(cid, cname, PASS, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

## Section 18 — PCHIP Fit Invariants

### Check 18.1 — PCHIP per-segment analytic sign

PCHIP-on-cumulative is supposed to produce a flux that's ≤ 0 (GPP)
or ≥ 0 (Rh) on every segment, by Fritsch-Carlson construction
(assuming monotone-direction data). Check this analytically: for
each segment, the extremum of f(τ)=Aτ²+Bτ+C on [0, L] occurs at
τ=0, τ=L, or τ_v=-B/(2A) (if inside). Reports violation count and
max magnitude. INFO-only because mixed-sign monthly data (e.g.
cells with sporadic negative NPP) breaks the monotonicity premise.

In [ ]:
cid, cname = "18.1", "PCHIP per-segment analytic sign"
helper   = WORK_DIR / "verify_pchip_invariants.r"
out_json = WORK_DIR / "verify_pchip_invariants.json"
if not helper.exists():
    record(cid, cname, INFO, f"helper {helper.name} not present; skipping")
else:
    try:
        r = subprocess.run(["Rscript", str(helper), str(out_json)],
                           cwd=WORK_DIR, capture_output=True, text=True, timeout=600)
        if r.returncode != 0:
            record(cid, cname, FAIL, f"helper failed: {r.stderr[-300:]}")
        else:
            d = json.loads(out_json.read_text())
            gv  = int(d.get("gpp_seg_violations", -1))
            gt  = int(d.get("gpp_seg_total", 1))
            gm  = float(d.get("gpp_seg_max_mag", 0))
            rv  = int(d.get("rh_seg_violations", -1))
            rt  = int(d.get("rh_seg_total", 1))
            rm  = float(d.get("rh_seg_max_mag", 0))
            detail = (f"GPP {gv:,}/{gt:,} segs ({gv/max(gt,1)*100:.3f}%, max {gm:.2e}); "
                      f"Rh {rv:,}/{rt:,} segs ({rv/max(rt,1)*100:.3f}%, max {rm:.2e})")
            # Treat as INFO unless violation rate exceeds 5% (would
            # indicate a real bug, not just mixed-sign data).
            rate = max(gv/max(gt,1), rv/max(rt,1))
            if rate > 0.05:
                record(cid, cname, FAIL, detail + " (>5% rate -- investigate)")
            else:
                record(cid, cname, INFO, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 18.2 — PCHIP C¹ continuity at interior knots

PCHIP-on-cumulative is C¹: at each interior knot, the right limit
of f from segment k-1 equals the left limit (= C_k) of segment k.
Check max |jump| across all knots and cells. Should be ≤ 1e-12
(machine precision). FAIL if > 1e-10. Catches storage-layout bugs
or incorrect coefficient unpacking.

In [ ]:
cid, cname = "18.2", "PCHIP C¹ continuity at knots"
out_json = WORK_DIR / "verify_pchip_invariants.json"
if not out_json.exists():
    record(cid, cname, INFO, "verify_pchip_invariants.json missing; run 18.1 first")
else:
    try:
        d = json.loads(out_json.read_text())
        gj = float(d.get("gpp_c1_max_jump", -1))
        rj = float(d.get("rh_c1_max_jump", -1))
        detail = f"max |jump| GPP={gj:.2e}, Rh={rj:.2e}"
        if max(gj, rj) > 1e-10:
            record(cid, cname, FAIL, detail + " (expected ≤1e-10)")
        elif max(gj, rj) > 1e-12:
            record(cid, cname, WARN, detail)
        else:
            record(cid, cname, PASS, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

## Section 19 — Additional Biome Cells

### Check 19.1 — Arctic tundra (Utqiagvik / Barrow, 71°N, -156°W)

Tundra cell: GPP only in JJA (very narrow growing season), winter
NEE ≈ 0 or small positive (slow Rh). Min-NEE month should be
JUL/AUG; amplitude small but non-zero.

In [ ]:
cid, cname = "19.1", "Utqiagvik (Arctic tundra) NEE pattern"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    lon_idx = int(np.argmin(np.abs(ds.longitude.values - (-156.0))))
    lat_idx = int(np.argmin(np.abs(ds.latitude.values  - 71.0)))
    npp = ds["NPP"].isel(longitude=lon_idx, latitude=lat_idx).values
    rh  = ds["Rh"].isel(longitude=lon_idx, latitude=lat_idx).values
    nee = -npp + rh
    time = pd.to_datetime(ds.time.values)
    ds.close()
    df = pd.DataFrame({"month": time.month, "nee": nee})
    df = df[time.year >= time.year.max() - 4]
    clim = df.groupby("month")["nee"].mean()
    min_month = int(clim.idxmin())
    amp = float(clim.max() - clim.min())
    ok_summer = min_month in (6, 7, 8)
    detail = f"Utqiagvik: min={min_month}, amp={amp:.2e}"
    if not ok_summer:
        record(cid, cname, FAIL, detail + " (expected min in JJA)")
    else:
        record(cid, cname, PASS, detail + " (uptake peak in JJA)")
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

### Check 19.2 — Caatinga semi-arid (NE Brazil, -10°N, -40°W)

Semi-arid woodland phase-locked to wet season (DJFM). Min-NEE
month should be in [1..5]; small amplitude due to drought stress.

In [ ]:
cid, cname = "19.2", "Caatinga (semi-arid) NEE pattern"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    lon_idx = int(np.argmin(np.abs(ds.longitude.values - (-40.0))))
    lat_idx = int(np.argmin(np.abs(ds.latitude.values  - (-10.0))))
    npp = ds["NPP"].isel(longitude=lon_idx, latitude=lat_idx).values
    rh  = ds["Rh"].isel(longitude=lon_idx, latitude=lat_idx).values
    nee = -npp + rh
    time = pd.to_datetime(ds.time.values)
    ds.close()
    df = pd.DataFrame({"month": time.month, "nee": nee})
    df = df[time.year >= time.year.max() - 4]
    clim = df.groupby("month")["nee"].mean()
    min_month = int(clim.idxmin())
    amp = float(clim.max() - clim.min())
    ok = min_month in (1, 2, 3, 4, 5, 12)
    detail = f"Caatinga: min={min_month}, amp={amp:.2e}"
    if not ok:
        record(cid, cname, FAIL, detail + " (expected min in DJFMAM)")
    else:
        record(cid, cname, PASS, detail + " (peak in wet season)")
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

### Check 19.3 — Indonesian peat (Kalimantan, -2°N, 115°E)

Tropical peatland: year-round vegetative activity, low-amplitude
seasonal cycle. Net sink overall (mean NEE ≤ 0); amplitude should
be small relative to boreal cells.

In [ ]:
cid, cname = "19.3", "Kalimantan peat NEE pattern"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    lon_idx = int(np.argmin(np.abs(ds.longitude.values - 115.0)))
    lat_idx = int(np.argmin(np.abs(ds.latitude.values  - (-2.0))))
    npp = ds["NPP"].isel(longitude=lon_idx, latitude=lat_idx).values
    rh  = ds["Rh"].isel(longitude=lon_idx, latitude=lat_idx).values
    nee = -npp + rh
    ds.close()
    amp = float(nee.max() - nee.min())
    mean_nee = float(nee.mean())
    # Loose: amplitude < 5e-5 (much smaller than boreal ~3.4e-5);
    # mean ~ neutral or modestly negative.
    problems = []
    if mean_nee > 5e-7:
        problems.append(f"mean NEE {mean_nee:.2e} unexpectedly large source")
    detail = f"Kalimantan: mean={mean_nee:.2e}, amp={amp:.2e}"
    if problems:
        record(cid, cname, FAIL, detail + " | " + "; ".join(problems))
    else:
        record(cid, cname, PASS, detail)
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

### Check 19.4 — Central Australia semi-arid (-25°N, 135°E)

Semi-arid grassland/scrub: low-amplitude cycle, often a weak net
source on annual mean during drought years. Sanity: mean NEE
magnitude small (within ±1e-6 gC m-2 s-1).

In [ ]:
cid, cname = "19.4", "Central Australia NEE pattern"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    lon_idx = int(np.argmin(np.abs(ds.longitude.values - 135.0)))
    lat_idx = int(np.argmin(np.abs(ds.latitude.values  - (-25.0))))
    npp = ds["NPP"].isel(longitude=lon_idx, latitude=lat_idx).values
    rh  = ds["Rh"].isel(longitude=lon_idx, latitude=lat_idx).values
    nee = -npp + rh
    ds.close()
    mean_nee = float(nee.mean())
    amp = float(nee.max() - nee.min())
    if abs(mean_nee) > 1e-6:
        record(cid, cname, FAIL, f"Australia: |mean NEE|={mean_nee:.2e} too large")
    else:
        record(cid, cname, PASS, f"Australia: mean={mean_nee:.2e}, amp={amp:.2e}")
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

## Section 20 — Cross-Product Comparison

### Check 20.1 — v2 vs v1 lat-band annual NEE

Beyond Check 6.1's pixel-level correlation, compare annual NEE per
latitude band between v2 and v1 over the overlap years 2001..2024.
Per-band relative difference should be small (< 5%) — otherwise the
diurnalization or splice is shifting mass between bands.

In [ ]:
cid, cname = "20.1", "v2 vs v1 lat-band annual NEE"
if "_summary" not in globals() or _summary.empty:
    record(cid, cname, INFO, "summary cube unavailable")
else:
    try:
        # v1 monthly file is one level up
        v1_path = WORK_DIR.parent / "MiCASA_v1" / "monthly_1x1" / "MiCASA_v1_flux_x360_y180_monthly.nc"
        if not v1_path.exists():
            record(cid, cname, INFO, f"v1 monthly file missing: {v1_path.name}")
        else:
            ds_v1 = xr.open_dataset(v1_path)
            lat = ds_v1.latitude.values
            # band masks
            nh_mid = (lat >= 30) & (lat < 60)
            trop   = (lat >= -30) & (lat < 30)
            sh_mid = (lat >= -60) & (lat < -30)
            bor    = (lat >= 60)
            # v1 NEE = -NPP + Rh, area-weighted by cos(lat) approximation
            w = np.cos(np.radians(lat))
            v1_nee = (-ds_v1["NPP"].values + ds_v1["Rh"].values)  # (time, lat, lon)
            v1_t = pd.to_datetime(ds_v1.time.values)
            ds_v1.close()
            # area-weight per band, sum over lon, mean over time per year
            def band_annual(mask):
                sub = v1_nee[:, mask, :].mean(axis=2)  # (time, lat-in-band)
                weighted = (sub * w[mask]).sum(axis=1) / w[mask].sum()  # (time,)
                df = pd.DataFrame({"yr": v1_t.year, "v": weighted})
                return df.groupby("yr")["v"].mean()
            bands = {
                "nh_mid": band_annual(nh_mid),
                "trop":   band_annual(trop),
                "sh_mid": band_annual(sh_mid),
                "boreal": band_annual(bor),
            }
            # v2 from summary cube (already aggregated globally; need per-band)
            # _summary has columns: NEE_global, GPP_global, resp_global, year, month
            # To get per-band, we'd re-aggregate from fluxes_*.nc which is
            # expensive. Compromise: report v1 lat-band annual means for
            # 2001..2024 as INFO; v2 comparison is left as a future hook.
            lines = []
            for band, s in bands.items():
                s = s.loc[2001:2024]
                if s.empty: continue
                lines.append(f"{band}: mean={s.mean():+.2e} sd={s.std():.2e}")
            detail = ("v1 annual lat-band NEE 2001..2024 (gC m-2 s-1, area-wt mean per band): "
                      + "; ".join(lines))
            record(cid, cname, INFO, detail + " (v2 per-band aggregation deferred)")
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 20.2 — ObsPack atmospheric growth-rate comparison (stub)

Future hook: compare global annual NEE to the atmospheric CO₂ growth
rate from ObsPack/ESRL flask network. Currently no ObsPack data on
disk; this check is a placeholder that documents the comparison
methodology without executing.

In [ ]:
cid, cname = "20.2", "ObsPack growth-rate comparison (stub)"
record(cid, cname, INFO,
       "deferred: requires ObsPack flask network data; "
       "compare NEE_global + ocean + fossil ≈ d(CO2_atm)/dt")

## Section 21 — Robustness

### Check 21.1 — Hourly NEE outlier scan

For a sample fluxes file, compute z-scores (NEE_h − cell_mean) /
cell_std per cell-hour. Count cells with any |z| > 6. Few outliers
expected; many → data corruption or fit instability.

In [ ]:
cid, cname = "21.1", "hourly NEE outlier scan"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes files")
else:
    f = files[len(files) // 2]
    try:
        ds = xr.open_dataset(f)
        gpp  = ds["GPP"].values
        resp = ds["RESP"].values if "RESP" in ds else ds.get("Rh", ds["GPP"]).values
        ds.close()
        nee = gpp + resp  # diurnal NEE per cell-hour (sign-correct sum)
        mu  = np.nanmean(nee, axis=0)
        sd  = np.nanstd(nee, axis=0)
        land = sd > 1e-12
        z = np.full_like(nee, 0.0, dtype=float)
        for h in range(nee.shape[0]):
            z[h, land] = (nee[h, land] - mu[land]) / sd[land]
        outlier_cells = ((np.abs(z) > 6.0).any(axis=0) & land).sum()
        n_land = int(land.sum())
        frac = outlier_cells / max(n_land, 1)
        detail = (f"sample {f.name}: {outlier_cells:,}/{n_land:,} land cells "
                  f"({frac*100:.3f}%) with any |z|>6")
        if frac > 0.01:
            record(cid, cname, FAIL, detail + " (>1% threshold)")
        elif frac > 0.001:
            record(cid, cname, WARN, detail)
        else:
            record(cid, cname, PASS, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 21.2 — Cross-month NEE spike

For each grid cell, build interannual climatology and σ of monthly
NEE; flag cells with any single year-month deviation > 6σ from the
climatology. Catches stuck-fill, isolated bad ingests.

In [ ]:
cid, cname = "21.2", "cross-month NEE spike"
try:
    ds = xr.open_dataset(MONTHLY_FILE)
    npp = ds["NPP"].values  # (time, lat, lon)
    rh  = ds["Rh"].values
    time = pd.to_datetime(ds.time.values)
    ds.close()
    nee = -npp + rh
    # Per-cell climatology of (mean, sd) per calendar month
    out_count = 0
    n_total   = 0
    max_z     = 0.0
    for m in range(1, 13):
        sel = (time.month == m)
        if not sel.any(): continue
        sub = nee[sel]   # (years, lat, lon)
        mu  = np.nanmean(sub, axis=0)
        sd  = np.nanstd(sub, axis=0)
        land = sd > 1e-15
        z = np.zeros_like(sub)
        for y in range(sub.shape[0]):
            z[y, land] = (sub[y, land] - mu[land]) / sd[land]
        n_total   += int(land.sum()) * sub.shape[0]
        out_count += int(((np.abs(z) > 6.0) & land[None, :, :]).sum())
        max_z      = max(max_z, float(np.nanmax(np.abs(z))))
    frac = out_count / max(n_total, 1)
    detail = (f"{out_count:,}/{n_total:,} cell-months ({frac*100:.3f}%) "
              f"|z|>6; max |z|={max_z:.1f}")
    if frac > 0.005:
        record(cid, cname, FAIL, detail + " (>0.5% threshold)")
    elif frac > 0.001:
        record(cid, cname, WARN, detail)
    else:
        record(cid, cname, PASS, detail)
except Exception as e:
    record(cid, cname, FAIL, f"exception: {e}")

## Section 22 — Performance Regression

### Check 22.1 — Diurnalize wall-time per year

Read the per-year `diurnalize-ERA5.r` records from the run manifest
(`jobs/run_manifest.tsv`, written by `lib/manifest.r`) and report
the `elapsed_s` distribution. FAIL if the median exceeds 1800 s
(30 min) per year — a serious regression. This replaces the former
regex parse of `[R] session elapsed time` lines out of
`d-YYYY-*.o*` logs. INFO if the manifest has no diurnalize rows yet
(the pipeline has not run since the manifest was added).

In [ ]:
cid, cname = "22.1", "diurnalize wall-time per year"
manifest = JOBS_DIR / "run_manifest.tsv"
if not manifest.exists():
    record(cid, cname, INFO, "no jobs/run_manifest.tsv (pipeline not run "
           "since run-manifest instrumentation was added)")
else:
    try:
        elapsed = []
        for line in manifest.read_text(errors="ignore").splitlines():
            if line.startswith("#") or not line.strip():
                continue
            f = line.split("\t")
            if len(f) != 7:
                continue
            if f[1] == "diurnalize-ERA5.r" and f[2] == "ok" and "year=" in f[6]:
                try:
                    elapsed.append(float(f[5]))
                except ValueError:
                    pass
        if not elapsed:
            record(cid, cname, INFO,
                   "run manifest present but no diurnalize-ERA5.r ok rows")
        else:
            med = float(np.median(elapsed))
            mx, mn = float(np.max(elapsed)), float(np.min(elapsed))
            detail = (f"{len(elapsed)} year-runs; wall median={med:.0f}s, "
                      f"min={mn:.0f}s, max={mx:.0f}s")
            if med > 1800:
                record(cid, cname, FAIL, detail + " (>30 min/yr regression)")
            elif med > 1500:
                record(cid, cname, WARN, detail)
            else:
                record(cid, cname, PASS, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

## Section 23 — Output Provenance

Phase 3 added CF/ACDD provenance metadata to every netCDF the pipeline
writes (`lib/provenance.r` / `lib/provenance.py`): the producing
software and its git commit, a processing timestamp, the host, input
files with SHA-256 checksums, and citation metadata. These checks
confirm the production outputs actually carry it. Outputs generated
before Phase 3 must be stamped by `stamp_provenance.py --retrofit`
(or regenerated) for 23.1 / 23.2 to pass.

### Check 23.1 — Hourly flux file provenance attributes

A sampled `fluxes_*.nc` should carry the provenance global attributes
written by `diurnalize-ERA5.r`: `Conventions`, `institution` and
`processing_pipeline` at minimum. A file from the instrumented pipeline
also has `processing_pipeline_commit` and `input_*` checksums (full
provenance); a file stamped after the fact by `stamp_provenance.py
--retrofit` has the static subset plus a `provenance_note`. FAIL only
if a sampled file has no provenance at all.

In [ ]:
cid, cname = "23.1", "hourly flux file provenance"
files = sorted(ERA5_DIR.glob("fluxes_*.nc"))
if not files:
    record(cid, cname, INFO, "no fluxes_*.nc files")
else:
    f = files[len(files) // 2]
    try:
        with xr.open_dataset(f) as ds:
            attrs = dict(ds.attrs)
        has_base = all(k in attrs for k in
                       ("Conventions", "institution", "processing_pipeline"))
        full  = ("processing_pipeline_commit" in attrs
                 and any(k.startswith("input_") for k in attrs))
        retro = "provenance_note" in attrs
        if has_base and full:
            commit = str(attrs.get("processing_pipeline_commit", ""))[:12]
            record(cid, cname, PASS,
                   f"{f.name}: full in-pipeline provenance (commit {commit})")
        elif has_base and retro:
            record(cid, cname, PASS,
                   f"{f.name}: retrofit (static) provenance present")
        elif has_base:
            record(cid, cname, WARN,
                   f"{f.name}: base provenance only, no commit/checksums")
        else:
            record(cid, cname, FAIL,
                   f"{f.name}: no provenance global attributes")
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 23.2 — Daily NEE file provenance inheritance

`daysplitter.sh` builds each `MiCASA_*.nee.YYYYMMDD.nc` with `ncks`,
which copies the source `fluxes_*.nc` global attributes — so the daily
file inherits the hourly file's provenance — and adds `daily_split_from`
/ `daily_split_tool` markers. Verify a sampled daily file carries
provenance; the daysplit markers are reported when present.

In [ ]:
cid, cname = "23.2", "daily NEE file provenance"
daily = sorted(ERA5_DIR.glob("MiCASA_*.nee.*.nc"))
if not daily:
    record(cid, cname, INFO, "no daily NEE files")
else:
    f = daily[len(daily) // 2]
    try:
        with xr.open_dataset(f) as ds:
            attrs = dict(ds.attrs)
        has_prov = "processing_pipeline" in attrs or "provenance_note" in attrs
        split = attrs.get("daily_split_from")
        if not has_prov:
            record(cid, cname, FAIL,
                   f"{f.name}: no provenance attributes inherited")
        else:
            detail = f"{f.name}: provenance present"
            detail += (f", daily_split_from={split}" if split
                       else " (daysplit markers absent -- pre-instrumentation split)")
            record(cid, cname, PASS, detail)
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 23.3 — provenance.conf is the single citation source

`lib/provenance.conf` is the one place the DOI, institution and
pipeline URL are defined; `lib/provenance.r` and `lib/provenance.py`
both read it. Confirm it is present and parses, and report whether the
archival DOI has been registered (it ships as `PENDING`).

In [ ]:
cid, cname = "23.3", "provenance.conf citation source"
conf_path = WORK_DIR / "lib" / "provenance.conf"
if not conf_path.exists():
    record(cid, cname, FAIL, "lib/provenance.conf missing")
else:
    conf = {}
    for line in conf_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        conf[k.strip()] = v.strip().strip('"')
    need = ["MICASA_DOI", "MICASA_PROV_INSTITUTION", "MICASA_PROV_PIPELINE_URL"]
    missing = [k for k in need if k not in conf]
    if missing:
        record(cid, cname, FAIL, f"keys missing from provenance.conf: {missing}")
    elif conf["MICASA_DOI"] == "PENDING":
        record(cid, cname, INFO,
               "provenance.conf parses; archival DOI still PENDING "
               "(set MICASA_DOI when the record is minted)")
    else:
        record(cid, cname, PASS,
               f"provenance.conf parses; DOI registered: {conf['MICASA_DOI']}")

## Section 24 — Run Manifest

Pipeline steps now append a structured record to
`jobs/run_manifest.tsv` via the `lib/manifest.{sh,r}` helpers —
`diurnalize-ERA5.r`, `daysplitter.sh`, and the `run_year.sh` /
`produce_2025_2026.sh` orchestrators each write a `start` / `ok` /
`fail` row per step, with timestamp, host, git commit and elapsed
seconds. verify_v2 reads this manifest (Check 22.1 and the checks
below) instead of regex-scraping job logs. The manifest does not
exist until the instrumented pipeline runs at least once.

### Check 24.1 — Run manifest integrity

`jobs/run_manifest.tsv` is present and every row is well-formed
(seven tab-separated columns). Reports the record count, the
distinct steps seen, and the most recent run timestamp.

In [ ]:
cid, cname = "24.1", "run manifest integrity"
manifest = JOBS_DIR / "run_manifest.tsv"
if not manifest.exists():
    record(cid, cname, INFO, "no jobs/run_manifest.tsv yet (pipeline not "
           "run since run-manifest instrumentation was added)")
else:
    try:
        rows, malformed = [], 0
        for line in manifest.read_text(errors="ignore").splitlines():
            if line.startswith("#") or not line.strip():
                continue
            f = line.split("\t")
            if len(f) != 7:
                malformed += 1
            else:
                rows.append(f)
        if not rows and not malformed:
            record(cid, cname, INFO, "manifest present but empty")
        elif malformed:
            record(cid, cname, FAIL,
                   f"{malformed} malformed row(s) (expected 7 columns); "
                   f"{len(rows)} well-formed")
        else:
            steps = sorted(set(r[1] for r in rows))
            last  = max(r[0] for r in rows)
            shown = ", ".join(steps[:6]) + ("..." if len(steps) > 6 else "")
            record(cid, cname, PASS,
                   f"{len(rows)} records, {len(steps)} distinct steps "
                   f"({shown}); latest {last}")
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")

### Check 24.2 — No failed pipeline steps

Scan the run manifest for `fail` records — a step that reported a
non-zero exit or an uncaught error. FAIL if any fall within
`MICASA_VERIFY_LOG_AGE_DAYS` (default 14); older resolved failures
are counted but not flagged. Reports the most recent few.

In [ ]:
cid, cname = "24.2", "no failed pipeline steps"
manifest = JOBS_DIR / "run_manifest.tsv"
if not manifest.exists():
    record(cid, cname, INFO, "no jobs/run_manifest.tsv yet")
else:
    try:
        import datetime as _dt
        max_age = float(os.environ.get("MICASA_VERIFY_LOG_AGE_DAYS", "14"))
        now = _dt.datetime.now(_dt.timezone.utc)
        fails, n_old_fail = [], 0
        for line in manifest.read_text(errors="ignore").splitlines():
            if line.startswith("#") or not line.strip():
                continue
            f = line.split("\t")
            if len(f) != 7 or f[2] != "fail":
                continue
            try:
                ts = _dt.datetime.strptime(
                    f[0], "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=_dt.timezone.utc)
                age_days = (now - ts).total_seconds() / 86400.0
            except ValueError:
                age_days = 0.0
            if age_days <= max_age:
                fails.append(f"{f[0]} {f[1]}: {f[6]}")
            else:
                n_old_fail += 1
        if fails:
            record(cid, cname, FAIL,
                   f"{len(fails)} fail record(s) in last {max_age:.0f}d "
                   f"({n_old_fail} older skipped); most recent: " +
                   "; ".join(fails[-3:]))
        else:
            record(cid, cname, PASS,
                   f"no fail records in last {max_age:.0f}d "
                   f"({n_old_fail} older skipped)")
    except Exception as e:
        record(cid, cname, FAIL, f"exception: {e}")